In [1]:
import urllib.error
import urllib.request
import pprint
from langchain.tools import tool

from langchain.chat_models import init_chat_model

import langchain_groq
import os

from dotenv import load_dotenv

load_dotenv()


True

In [2]:

model_groq_lamma70b = init_chat_model("llama-3.3-70b-versatile",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=10000, temperature=0.0)

# model = init_chat_model("openai/gpt-4o-mini",
#                         api_key=os.environ["OPENROUTER_API_KEY"],
#                         model_provider="openrouter",
#                         base_url="https://openrouter.ai/api/v1",
#                         max_tokens=1000, temperature=0.0)

# model = init_chat_model("nvidia/nemotron-3-ultra-550b-a55b:free",
#                         api_key=os.environ["OPENROUTER_API_KEY"],
#                         model_provider="openrouter",
#                         base_url="https://openrouter.ai/api/v1",
#                         max_tokens=1000, temperature=0.0)


# model = init_chat_model("openrouter/free",
#                         api_key=os.environ["OPENROUTER_API_KEY"],
#                         model_provider="openrouter",
#                         base_url="https://openrouter.ai/api/v1",
#                         max_tokens=1000, temperature=0.0)




In [3]:
model_or = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)


In [4]:
model_or_paid_gpt56_luna_pro = init_chat_model("openai/gpt-5.6-luna-pro",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)


In [ ]:
# response = model.invoke("which model are you?")

# pprint.pprint("Model response:")
# pprint.pprint(response)

'Model response:'
AIMessage(content='I am a Meta AI model, and my specific model name is Llama. Llama stands for "Large Language Model Meta AI."', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 40, 'total_tokens': 68, 'completion_time': 0.113340045, 'completion_tokens_details': None, 'prompt_time': 0.001913595, 'prompt_tokens_details': None, 'queue_time': 0.058499274, 'total_time': 0.11525364}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fdf63-728f-7f40-b426-d94102dd2cc5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 28, 'total_tokens': 68})


In [5]:
from langchain_core.tools import tool
import sqlite3

In [6]:

@tool
def save_trip_demo(user_id: str, destination: str) -> str:
    """Save a trip to the database. Irreversible without manual cleanup."""
    return f"Trip to {destination} saved for {user_id}."  # standing in for a real DB write

In [ ]:
from langchain.agents import create_agent

from langchain.agents.middleware import SummarizationMiddleware


agent = create_agent(
    model=model_or_paid_gpt56_luna_pro,
    tools=[save_trip_demo],
    middleware=[
        SummarizationMiddleware(
            model = model_groq_lamma70b,
            trigger = ("tokens",50),
            keep =('messages',10),
        )
    ])

## HITL middleware

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def your_read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def your_send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model_groq_lamma70b,
    tools=[your_read_email_tool, your_send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "your_send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "your_read_email_tool": False,
            }
        ),
    ],
)

In [9]:
config = {'configurable':{"thread_id":"hitl"}}

result =agent.invoke({"messages": [("user", "Send an email to my manager on sunjtry@gmail.com, asking for a leave")]}, config=config)

In [11]:
from rich import print as rprint

In [12]:
rprint(result)

{
    'messages': [
        HumanMessage(
            content='Send an email to my manager on sunjtry@gmail.com, asking for a leave',
            additional_kwargs={},
            response_metadata={},
            id='287afc01-c613-4437-b4e0-43480a2de059'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'tool_calls': [
                    {
                        'id': '0m7yvww0y',
                        'function': {
                            'arguments': '{"body":"Dear Manager, I am writing to request a leave. Thank 
you.","recipient":"sunjtry@gmail.com","subject":"Leave Request"}',
                            'name': 'your_send_email_tool'
                        },
                        'type': 'function'
                    }
                ]
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 46,
                    'prompt_tokens': 312,
                    'total_tokens': 358,
                    'completion_time': 0.105481161,
                    'completion_tokens_details': None,
                    'prompt_time': 0.016762344,
                    'prompt_tokens_details': None,
                    'queue_time': 0.16118751,
                    'total_time': 0.122243505
                },
                'model_name': 'llama-3.3-70b-versatile',
                'system_fingerprint': 'fp_3272ea2d91',
                'service_tier': 'on_demand',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'groq'
            },
            id='lc_run--01a007e7-e09d-7460-869e-3a322999ec00-0',
            tool_calls=[
                {
                    'name': 'your_send_email_tool',
                    'args': {
                        'body': 'Dear Manager, I am writing to request a leave. Thank you.',
                        'recipient': 'sunjtry@gmail.com',
                        'subject': 'Leave Request'
                    },
                    'id': '0m7yvww0y',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={'input_tokens': 312, 'output_tokens': 46, 'total_tokens': 358}
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'your_send_email_tool',
                        'args': {
                            'body': 'Dear Manager, I am writing to request a leave. Thank you.',
                            'recipient': 'sunjtry@gmail.com',
                            'subject': 'Leave Request'
                        },
                        'description': "Tool execution requires approval\n\nTool: your_send_email_tool\nArgs: 
{'body': 'Dear Manager, I am writing to request a leave. Thank you.', 'recipient': 'sunjtry@gmail.com', 'subject': 
'Leave Request'}"
                    }
                ],
                'review_configs': [
                    {'action_name': 'your_send_email_tool', 'allowed_decisions': ['approve', 'edit', 'reject']}
                ]
            },
            id='a7166542fb9848a316a653357471befb'
        )
    ]
}

In [19]:
result

{'messages': [HumanMessage(content='Send an email to my manager on sunjtry@gmail.com, asking for a leave', additional_kwargs={}, response_metadata={}, id='07902d44-9ed9-43c4-8047-d983b71aa4b5'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ydh6sg0rd', 'function': {'arguments': '{"body":"Dear Manager, I am writing to request a leave. Thank you.","recipient":"sunjtry@gmail.com","subject":"Leave Request"}', 'name': 'your_send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 312, 'total_tokens': 358, 'completion_time': 0.112307915, 'completion_tokens_details': None, 'prompt_time': 0.016317767, 'prompt_tokens_details': None, 'queue_time': 0.051488213, 'total_time': 0.128625682}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fdf72-292c-7613-b836-dd6ea50158

In [13]:
# --- Core LangChain ---
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.tools import tool as tool_rt, ToolRuntime

In [16]:
# --- LangGraph (checkpointing, resuming) ---
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

In [17]:
from langchain.agents.middleware import (
    SummarizationMiddleware,
    HumanInTheLoopMiddleware,
    ModelCallLimitMiddleware,
    ToolCallLimitMiddleware,
    ModelFallbackMiddleware,
    PIIMiddleware,
    TodoListMiddleware,
    LLMToolSelectorMiddleware,
    ToolRetryMiddleware,
    ModelRetryMiddleware,
    LLMToolEmulator,
    ContextEditingMiddleware,
    ClearToolUsesEdit,
)

Defining Tools for my Cinebot - Dummy as of now but ofcourse will be real as we have seen in last class.

In [18]:
@tool
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie at the cinema."""
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM only",
        "oppenheimer": "Sold out for tonight",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")

In [19]:

@tool
def book_seats(movie_title: str, seat_count: int) -> str:
    """Book seats for a movie. Irreversible once confirmed."""
    return f"Booked {seat_count} seat(s) for {movie_title}."

In [20]:
@tool
def cancel_booking(booking_id: str) -> str:
    """Cancel an existing booking. Irreversible."""
    return f"Booking {booking_id} cancelled."

In [21]:
@tool
def check_order_status(booking_id: str) -> str:
    """Check the status of an existing booking."""
    return f"Booking {booking_id}: confirmed, 2 seats, Interstellar, 7:00 PM."

In [22]:
@tool
def get_refund_policy() -> str:
    """Get the cinema's refund policy -- exact wording, not to be paraphrased."""
    return "Refunds available up to 2 hours before showtime. No refunds after that."

In [23]:

@tool
def lookup_seat_map(movie_title: str, seat_number: str) -> str:
    """Look up a specific seat -- fails if the seat number format is wrong."""
    if not seat_number or not seat_number[0].isalpha():
        raise ValueError(f"Malformed seat number '{seat_number}' -- expected a letter+number like 'A12'.")
    return f"Seat {seat_number} for {movie_title}: available."

In [24]:
cinebot_tools = [check_showtimes, book_seats, cancel_booking, check_order_status, get_refund_policy, lookup_seat_map]


### Summarization Middleware

In [25]:
summarizing_agent = create_agent(
    model=model_or_paid_gpt56_luna_pro,
    tools=cinebot_tools,
    middleware=[
        SummarizationMiddleware(
            model=model_or,
            trigger=('tokens', 80),
            keep=('messages', 3),
        )
    ]
)

In [26]:
result = summarizing_agent.invoke({"messages": [("user", "Is Interstellar showing tonight? also please make sure that you book me a ticket, refund me if it is not available,also share the refund policy for me to go through, also check my order status for book_1234")]})

In [27]:
rprint(result)

{
    'messages': [
        HumanMessage(
            content='Here is a summary of the conversation to date:\n\nUser Safety: safe',
            additional_kwargs={'lc_source': 'summarization'},
            response_metadata={},
            id='ad4273c4-eddb-4374-b9a8-6be929e93a74'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_content': "**Checking booking process**\n\nI need to check the showtimes, ticket 
policies, and order status simultaneously since booking is only possible if seats are available. The user seems to 
want to book probably one ticket. I should first check the availability before confirming the booking since it's 
irreversible. If a ticket isn't available, I might need to clarify that I can't issue a refund unless there’s an 
existing booking. I’ll make sure to explain that clearly.",
                'reasoning_details': [
                    {
                        'summary': "**Checking booking process**\n\nI need to check the showtimes, ticket policies,
and order status simultaneously since booking is only possible if seats are available. The user seems to want to 
book probably one ticket. I should first check the availability before confirming the booking since it's 
irreversible. If a ticket isn't available, I might need to clarify that I can't issue a refund unless there’s an 
existing booking. I’ll make sure to explain that clearly.",
                        'type': 'reasoning.summary',
                        'format': 'openai-responses-v1',
                        'index': 0
                    },
                    {
                        'data': 
'gAAAAABqgQJ8QCREPm-enV63ed3308wSygIim97MfRHKf5VPkj2djv-3nC7EuBsOGgwY7mfGUW6VJ9q5kCd3wVPj3UF__K0MbUXVzikixhF1wUKMBd
O-VPtLfssRN7y0cCSANYNvl89bWuo29_tqdMJDvTihI5s11KLIn2a4kFLZ_0K3gofwuJehchjRSUKkRunPufesAjGoyBD2CRmIUtBz9lnShVU3lU1IQ
MzCZr_fRPOelPDCkr7CjyPFvx5vluaL8in_l538SYzmepQeycRdAbqszsq6L3R-iuekLZ1TtFyi7EM3V1kHhL0iTG9gbghTZbb6NWPbJ3Yv1sT-qWlf
W-Xc6Z2Rl1R6Adplpyu0YMAEBOeHf0I6fhTmrzcSDb4ailLDKKcsqu5bq-Ocx2WObOcAUjPLi5IUPyeD7lk6I6jN8-IMDPdNO0HJ4uJR8jtm-gQ3r9T
kPZsmDzvp7HF-biqU81iiAAexpOipCutU0wvRVbZcvCD2T_HZMaxpn0MDBk_UHiybg6Y7v-OTQsMe046VBEQb0Vt-uYAHmJPMqKPN1z9uRVrvn0bOw_
7kgnocSN_rYXIxvCJ_S_FVDatNpu8L_L4dcsHt4LhZNPtfSTqR4Wfs4MUK3I27VGTKKyWSSQRTT-g_ApkDM1WfD42uCEn61PCPID18QKEHLvNfQSNM9
RGWBOAL0TBYTitQCmKGFcSmOL3MvdMQgjrjPZAPWtY5J4TLUxmNDqOEIztMFkOerZUuJZGgn583W7snImIXz7Hf4-R-v0DF3Zb0Xsge7fmS1AHsOF_p
zqhP-lOWgW_WIpu8dFeg2iDaiLwI9npo2TPuSsOQSj84kz7f5l50doCBBgIQM7Q1Jz7BPn9wTUjPsIboc6acApZZwiwDKjkSytkXlnuxELnZcdiAeg4
_y_xUw0j9ed3e_u8N5_TOAAmB6B9Dav2lV9ScU-oCQc40kaduHUXgrrNsYPG3yEFkroenO0AlStqJfLu_vHhzJc13I7FbsQb2eF6mQwLENFO3tTXuG3
xUBz--MbJB4fuI6135P_C2bkQM9E2k2nE0cdsBwcDr1gKAO3LOVun_KFCpVr651uikFvPOUN-bYDtDYRbuaBXD-N0YnfUWVTVDkmDetmwwyg3ptaRJ8
HOHN3PHFnQsvnbm3sUxaeiMtjJKfp2XpiQOncPIQEDfIbPKkI-RU5Q0BtuBfXYTc1VvJ01J1-wPV5l0Z7g5gIYUd6E-7MDCvkcM_T9DNvQJxf1AAUPr
AzzKeXbJ96xaSTjIf-6eIAQ92Gb97wa7cMTp9-BOaqGhJwqomUxCOs_7FMRSgcdGktPLXK9f_MnE5-hn8iTY3O2GvEYbbD30EwcYagc_7Hx9Vz4_dhl
2tIB9BME7qydLK3aATxJnHv4QddFCroJhVStrGKnVhZMFxiJPRWbdUPgEBhRcsxhM75X-voVgElpuoSVv0njolE07YQXEIiNbkkZb96AObZscwBpKyE
8BaICpP85BwlWOKQAP_k5ekUM5SOZpGS-Hs_F2jHSXWWADT6rtJOv085xOqur--rrWMQ0Kfy-cOlBgTXNvRBjyRGsb_NS7mexK5Vg0C6IM5PDOza7kF
GESvo_ZeVDjFPtnJdXnhPblF2k5AmhlOOCU8F1kUSN-RbLVL-RN6kgw2CQSUbKZVlg8jofB_Xha49sosF2EvSFJGkx19JTqMyo0oSwAmGG8hcYjbtTB
dRPSPTQsjH-KQC4uPeWqDcjvQ3is93-xmHgjd7cjQEC-YxshTSfuiYBkB3HqzmBRC0Rj5mNNFwNuqtRp.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2
dwdC01LjYtbHVuYS1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_07d4c559e72bca93016a81027b55f88195a2bf7c1fa10d35c8',
                        'index': 1
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786839670-GEQ9

# HITL (Human in the Loop)

In [28]:
guarded_agent = create_agent(
    model=model_or,
    tools=cinebot_tools,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"cancel_booking": {"allowed_decisions": ["approve", "edit", "reject", "respond"]}}
        ),
    ],
    checkpointer=InMemorySaver(),  # REQUIRED -- HITL needs to pause and later resume
)

config = {'configurable':{'thread_id':'hitl-demo-live'}}

In [29]:
result = guarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)

In [30]:
rprint(result)

{
    'messages': [
        HumanMessage(
            content='Please cancel booking BK1042',
            additional_kwargs={},
            response_metadata={},
            id='15cdb4b8-6593-4a02-802b-8bb9b1de7dc0'
        ),
        AIMessage(
            content="I'll cancel your booking BK1042 right away.",
            additional_kwargs={},
            response_metadata={
                'model_name': 'poolside/laguna-s-2.1:free',
                'id': 'gen-1786839794-buGBMv1qhzong7ByKKm9',
                'created': 1786839794,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--01a007f3-7c8a-7402-aa89-f2aa1bdbebc4-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'BK1042'},
                    'id': 'chatcmpl-tool-ee9454865b414f39a56b558fabe26133',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 482,
                'output_tokens': 41,
                'total_tokens': 523,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'cancel_booking',
                        'args': {'booking_id': 'BK1042'},
                        'description': "Tool execution requires approval\n\nTool: cancel_booking\nArgs: 
{'booking_id': 'BK1042'}"
                    }
                ],
                'review_configs': [
                    {
                        'action_name': 'cancel_booking',
                        'allowed_decisions': ['approve', 'edit', 'reject', 'respond']
                    }
                ]
            },
            id='46746f1f5c844ff1c709404409c2c41f'
        )
    ]
}

In [42]:
from rich import print

In [43]:
print(result)

{
    'messages': [
        HumanMessage(
            content='Please cancel booking BK1042',
            additional_kwargs={},
            response_metadata={},
            id='d9762a95-5c3a-4aa4-95f9-e5b962c20eb0'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_content': 'We need to call cancel_booking function.',
                'reasoning_details': [
                    {
                        'type': 'reasoning.text',
                        'format': 'unknown',
                        'index': 0,
                        'text': 'We need to call cancel_booking function.'
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-oss-20b:free',
                'id': 'gen-1786165279-qUgA4qHpnZ8rVZLxVg3V',
                'created': 1786165279,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 2.45e-06,
                    'upstream_inference_prompt_cost': 4.0745e-06,
                    'upstream_inference_cost': 6.5245e-06
                }
            },
            id='lc_run--019fdfbf-3971-7d40-a6a8-9008dd189b8c-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'BK1042'},
                    'id': 'call_577C7D30179D4BD68E8D88A5',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 281,
                'output_tokens': 35,
                'total_tokens': 316,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 8}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'cancel_booking',
                        'args': {'booking_id': 'BK1042'},
                        'description': "Tool execution requires approval\n\nTool: cancel_booking\nArgs: 
{'booking_id': 'BK1042'}"
                    }
                ],
                'review_configs': [
                    {
                        'action_name': 'cancel_booking',
                        'allowed_decisions': ['approve', 'edit', 'reject', 'respond']
                    }
                ]
            },
            id='f08be289e18e4a510e40dc4179382ed0'
        )
    ]
}

## HITL Decision to resume agent

In [31]:
resumed_result = guarded_agent.invoke(Command(resume={"decisions":[{"type":"approve"}]}),config=config)

In [33]:
rprint(resumed_result)

{
    'messages': [
        HumanMessage(
            content='Please cancel booking BK1042',
            additional_kwargs={},
            response_metadata={},
            id='15cdb4b8-6593-4a02-802b-8bb9b1de7dc0'
        ),
        AIMessage(
            content="I'll cancel your booking BK1042 right away.",
            additional_kwargs={},
            response_metadata={
                'model_name': 'poolside/laguna-s-2.1:free',
                'id': 'gen-1786839794-buGBMv1qhzong7ByKKm9',
                'created': 1786839794,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--01a007f3-7c8a-7402-aa89-f2aa1bdbebc4-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'BK1042'},
                    'id': 'chatcmpl-tool-ee9454865b414f39a56b558fabe26133',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 482,
                'output_tokens': 41,
                'total_tokens': 523,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Booking BK1042 cancelled.',
            name='cancel_booking',
            id='015da808-7a67-440f-b0f8-4591fbb58ba4',
            tool_call_id='chatcmpl-tool-ee9454865b414f39a56b558fabe26133'
        ),
        AIMessage(
            content='Your booking BK1042 has been successfully cancelled.',
            additional_kwargs={
                'reasoning_content': 'The booking has been successfully cancelled. I should inform the user.',
                'reasoning_details': [
                    {
                        'type': 'reasoning.text',
                        'format': 'unknown',
                        'index': 0,
                        'text': 'The booking has been successfully cancelled. I should inform the user.'
                    }
                ]
            },
            response_metadata={
                'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free',
                'id': 'gen-1786839820-vEtz9L2uUYTgTPDdpsVO',
                'created': 1786839820,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--01a007f3-e199-7291-950e-6d832355dd2f-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 719,
                'output_tokens': 27,
                'total_tokens': 746,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 18}
            }
        )
    ]
}

In [42]:
def run_interactive_hitl_demo(agent, config):
    """A genuinely interactive HITL loop -- ask out loud, type the answer, watch it apply live."""
    state = agent.get_state(config)
    if not state.next:
        print("Nothing is currently paused for approval.")
        return

    print("The agent wants to call a guarded tool. Choose a decision:")
    print("  1) approve  -- run it exactly as proposed")
    print("  2) edit     -- run it, but change the booking_id first")
    print("  3) reject   -- block it, with a reason sent back to the agent")
    print("  4) respond  -- answer a question instead of deciding on the action")

    choice = input("Type 1, 2, 3, or 4: ").strip()

    if choice == "1":
        decision = {"type": "approve"}
    elif choice == "2":
        new_id = input("New booking_id to use instead: ").strip()
        decision = {"type": "edit", "args": {"booking_id": new_id}}
    elif choice == "3":
        reason = input("Reason for rejecting: ").strip()
        decision = {"type": "reject", "message": reason}
    elif choice == "4":
        answer = input("Your response to the agent: ").strip()
        decision = {"type": "respond", "message": answer}
    else:
        print("Not a valid choice -- try again.")
        return

    resumed = agent.invoke(Command(resume={"decisions": [decision]}), config=config)
    print()
    print("Agent's final response:", resumed["messages"][-1].content)
    rprint(resumed)

In [43]:
config = {'configurable':{'thread_id':'hitl-demo-live-2'}}

In [44]:
newguarded_agent = create_agent(
    model=model_or_paid_gpt56_luna_pro,
    tools=cinebot_tools,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"cancel_booking": {"allowed_decisions": ["approve", "edit", "reject", "respond"]}}
        ),
    ],
    checkpointer=InMemorySaver(),  # REQUIRED -- HITL needs to pause and later resume
)

In [45]:
newresult = newguarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)

In [46]:
run_interactive_hitl_demo(newguarded_agent, config)

The agent wants to call a guarded tool. Choose a decision:
  1) approve  -- run it exactly as proposed
  2) edit     -- run it, but change the booking_id first
  3) reject   -- block it, with a reason sent back to the agent
  4) respond  -- answer a question instead of deciding on the action

Agent's final response: The cancellation was not completed. Booking **BK1042** remains active.


{
    'messages': [
        HumanMessage(
            content='Please cancel booking BK1042',
            additional_kwargs={},
            response_metadata={},
            id='45ea3f13-802c-458d-8d7a-99ce6f5a1643'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'data': 
'gAAAAABqgQP7fE2KTzHnzJF5TUEDBvjpDr0D0jO-QWIzkWQOrpIXRjAW5vJeE_7jLziYYxmQTQOq8O32kLLIzJx68i8udwp3NyQQWMo7JflUfbpeT_
t2mC4butmLxT6C4i5E9zZOrs7qUbjFtHNZwbR2hAqPzCmAxkTjE5BTcIN0MxnPYrxFW-gta1PcnvZHkMByPCCFzE5zmgqPDb2osVFiy_2aZul8IfQHA
3rIJxwRTDCBMUL1vjXbZ_Z907KhRYfAe0mEluZN2amsOw_Ra0tdgyucWeKb_es2_XYngNluBA-J-ZThDoHvzg8FjdluzPO3lEG6_kzptjMRGI7WEQOI
HlTS_kKldEuvNZZ5ZAvV4w1P3oZOVPVRJgElZLl7xVf8zLGI6t9g1NrV8ie00e1-QXpQAaOiw1p356NCCu404qBVxLTPqGiXmptYqvYOfLyMDFMoRJW
Zp-kXPMCnEuyPw0UkPqtcnPX2pXgtn-tkv6fNIpsR6d_Bak8e01a183-fR379DZKe0iEiV5COA2OlU4VEN30DB0RKnQogfDmPWYq8_FDROGyqYlbLAS
w-BaItDfPomtFc6yruhNu0e6k7jVy4JxuXAPPSxmxPMVgQykPt_tjuiGLVi1SgJael_5HnjBK8sg7bMqYumdXZsAR_1xYYlgzRESRsWWvQFyh1ChpIh
CW0LKKAUxzOvU9Q_DxUPP2SgvsqdEyutVFo4JQNPa6aen9ZAfxE_sX5Gu5QBH9E2C8s9DGEFUbPScU_j8JmbhIuVUl7LOb-QRsj6hQX9zy_kXXJTWMP
wyS0rCGdfj9aw-W7LSw5JQyoMN9VZkwxO0Cd0daxdVWkF8bj6O6GF9_asOnHt3lJdOJBBrGE4OiN05dOLCkDNzIFz_bvUS-KrYtKvtHwQKixihjXPue
FfaJlGmQlzn7zAatj_BlnleGMv1ZFImDKoyooRQPpOQOGsx533UQbJUR63GVv1K8c_X9x3VIk3_cOqzp57PEeaAICkTl_w_Etif73Uj658ac8yyWENK
32zV0h1Gfbn-jjZPfieiPBIZA2tJ0xGq56eGilMSNo4PBohK6sxlUk6OV2SQSBCVYkej7V6fUtMtcATRj2fAmK12lngiWcNG3u7niM3fWkDHCThkT1U
iFPUb2ULfhiI2mCKe9a1pBNyVarKcXI3LOq_3NFKWsYe78IBX3jardyMsSGk8M1klgFz4rwZ8tE2L941bHLGRwhF_V1SuwP1XqlyX657NUEGWnZwpf3
OmObz9u9xcKLKFgTsWM8wAD3TlCTkFYMICQenfmO6Ano6T4XBBSsBhWrYN7w4Q_nXbdhHkqg8N_waJcuCBTZ-AfmZC8Y_28wDVPGewYSaiuMLIX7qpz
7zdmRndX508O2TKirtdQ4NRVJ7H_5D9Hp48X0kNM1z2nfS-RCEaubwJtdPH7cDw==.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuY
S1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_041bf4afc0e9730a016a8103fb658c8195ae71f86a750c3c90',
                        'index': 0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786840056-DBlJp0cKVHVEhYwW91Pe',
                'created': 1786840056,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0003372,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.000123,
                    'upstream_inference_prompt_cost': 0.0002142,
                    'upstream_inference_cost': 0.0003372
                }
            },
            id='lc_run--01a007f7-7c39-7f73-9069-d31ce2958dd7-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'BK1042'},
                    'id': 'call_NMgOyxYl2IPLf8hvaeLb5Pt5',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 2142,
                'output_tokens': 205,
                'total_tokens': 2347,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 119}
            }
        ),
        ToolMessage(
            content='I donot want to cancel as my friend can go',
            name='cancel_booking',
            id='c9dd4eab-83e7-439b-a345-a236b995a17a',
            tool_call_id='call_NMgOyxYl2IPLf8hvaeLb5Pt5',
            status='error'
        ),
        AIMessage(
            content='The cancellation was not completed. Booking **BK1042** remains active.',
            additional_kwargs={
                're

⚙️ Code Walkthrough: Command(resume={"decisions": [...]}) is how you hand a decision back to an agent that's paused mid-run. The decisions list has one entry per interrupted tool call (usually just one). Each decision is a dict with a "type" key matching one of the four options, plus whatever extra data that type needs — edit needs new args, reject and respond need a message, approve needs nothing else.

# Model Call Limit

In [58]:
call_limited_agent = create_agent(
    model=model_or,
    tools=cinebot_tools,
    checkpointer=InMemorySaver(),  # required for thread_limit to persist across calls
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=5,   # across the WHOLE conversation
            run_limit=1,       # per single .invoke() call
            exit_behavior="end",  # graceful stop, not an exception
        ),
    ],
)

In [59]:
result = call_limited_agent.invoke(
    {"messages": [("user", "What's showing tonight?")]},
    config={"configurable": {"thread_id": "call-limit-demo"}},
)

In [60]:
print(result)

{
    'messages': [
        HumanMessage(
            content="What's showing tonight?",
            additional_kwargs={},
            response_metadata={},
            id='48ac015e-92e6-49db-9416-726f1cea35ab'
        ),
        AIMessage(
            content="I'd be happy to check what's showing tonight! However, I need to know which specific movie 
you're interested in to look up the showtimes. \n\nCould you let me know the name of the movie you'd like to see? 
Or if you're not sure what's playing, you might want to check the cinema's website or app for a full listing of 
tonight's showtimes.",
            additional_kwargs={},
            response_metadata={
                'model_name': 'poolside/laguna-xs-2.1:free',
                'id': 'gen-1786167936-JkDF6lLtz0WqkM0b7SNG',
                'created': 1786167936,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--019fdfe7-c231-7d23-a9ef-c1cfee66e226-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 472,
                'output_tokens': 81,
                'total_tokens': 553,
                'input_token_details': {'cache_read': 16, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        )
    ]
}

In [61]:
result_invoke_2= call_limited_agent.invoke(
    {"messages": [("user", "cancel my booking B123? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)
print(result_invoke_2)

{
    'messages': [
        HumanMessage(
            content='cancel my booking B123? ',
            additional_kwargs={},
            response_metadata={},
            id='e92f2dfb-c7d4-4292-8363-88ecd1ff9208'
        ),
        AIMessage(
            content="I'll cancel your booking B123 for you.",
            additional_kwargs={},
            response_metadata={
                'model_name': 'poolside/laguna-xs-2.1:free',
                'id': 'gen-1786168351-1GT40QPuqQCoX01ydPnn',
                'created': 1786168351,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--019fdfee-18d9-73d1-b1a8-07f6c92cb95a-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B123'},
                    'id': 'chatcmpl-tool-bdbb506da49c7768',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 477,
                'output_tokens': 38,
                'total_tokens': 515,
                'input_token_details': {'cache_read': 16, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Booking B123 cancelled.',
            name='cancel_booking',
            id='fdfe47ea-e0f7-4a90-a55b-7e2e7aed5b78',
            tool_call_id='chatcmpl-tool-bdbb506da49c7768'
        ),
        AIMessage(
            content='Model call limits exceeded: run limit (1/1)',
            additional_kwargs={},
            response_metadata={},
            id='af8f1d44-fe9b-4913-9fd8-ba34166efbdc',
            tool_calls=[],
            invalid_tool_calls=[]
        )
    ]
}

In [62]:
result_invoke3 = call_limited_agent.invoke(
    {"messages": [("user", "What all movies are being shown? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)
print(result_invoke3)

{
    'messages': [
        HumanMessage(
            content='cancel my booking B123? ',
            additional_kwargs={},
            response_metadata={},
            id='e92f2dfb-c7d4-4292-8363-88ecd1ff9208'
        ),
        AIMessage(
            content="I'll cancel your booking B123 for you.",
            additional_kwargs={},
            response_metadata={
                'model_name': 'poolside/laguna-xs-2.1:free',
                'id': 'gen-1786168351-1GT40QPuqQCoX01ydPnn',
                'created': 1786168351,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--019fdfee-18d9-73d1-b1a8-07f6c92cb95a-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B123'},
                    'id': 'chatcmpl-tool-bdbb506da49c7768',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 477,
                'output_tokens': 38,
                'total_tokens': 515,
                'input_token_details': {'cache_creation': 0, 'cache_read': 16},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Booking B123 cancelled.',
            name='cancel_booking',
            id='fdfe47ea-e0f7-4a90-a55b-7e2e7aed5b78',
            tool_call_id='chatcmpl-tool-bdbb506da49c7768'
        ),
        AIMessage(
            content='Model call limits exceeded: run limit (1/1)',
            additional_kwargs={},
            response_metadata={},
            id='af8f1d44-fe9b-4913-9fd8-ba34166efbdc',
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(
            content='What all movies are being shown? ',
            additional_kwargs={},
            response_metadata={},
            id='73e19192-ca69-48c3-88c1-5595edf3a4ed'
        ),
        AIMessage(
            content="I don't have a function to list all movies currently being shown at the cinema. My available 
tools only allow me to check showtimes for a specific movie when you provide its title.\n\nTo find out what movies 
are playing, you might need to:\n- Check the cinema's website or app directly\n- Visit the cinema's lobby or box 
office\n- Ask me about a specific movie you're interested in, and I can check its showtimes\n\nIs there a 
particular movie you'd like to know the showtimes for?",
            additional_kwargs={},
            response_metadata={
                'model_name': 'poolside/laguna-xs-2.1:free',
                'id': 'gen-1786168375-mBJxHtB7cE4qGCmyM8ZH',
                'created': 1786168375,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--019fdfee-7694-7940-a42d-e0372660e397-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 564,
                'output_tokens': 108,
                'total_tokens': 672,
                'input_token_details': {'cache_read': 16, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        )
    ]
}

In [63]:
result_invoke4 = call_limited_agent.invoke(
    {"messages": [("user", "Summarize my chat? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)

In [64]:
print(result_invoke4)

{
    'messages': [
        HumanMessage(
            content='cancel my booking B123? ',
            additional_kwargs={},
            response_metadata={},
            id='e92f2dfb-c7d4-4292-8363-88ecd1ff9208'
        ),
        AIMessage(
            content="I'll cancel your booking B123 for you.",
            additional_kwargs={},
            response_metadata={
                'model_name': 'poolside/laguna-xs-2.1:free',
                'id': 'gen-1786168351-1GT40QPuqQCoX01ydPnn',
                'created': 1786168351,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--019fdfee-18d9-73d1-b1a8-07f6c92cb95a-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B123'},
                    'id': 'chatcmpl-tool-bdbb506da49c7768',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 477,
                'output_tokens': 38,
                'total_tokens': 515,
                'input_token_details': {'cache_creation': 0, 'cache_read': 16},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Booking B123 cancelled.',
            name='cancel_booking',
            id='fdfe47ea-e0f7-4a90-a55b-7e2e7aed5b78',
            tool_call_id='chatcmpl-tool-bdbb506da49c7768'
        ),
        AIMessage(
            content='Model call limits exceeded: run limit (1/1)',
            additional_kwargs={},
            response_metadata={},
            id='af8f1d44-fe9b-4913-9fd8-ba34166efbdc',
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(
            content='What all movies are being shown? ',
            additional_kwargs={},
            response_metadata={},
            id='73e19192-ca69-48c3-88c1-5595edf3a4ed'
        ),
        AIMessage(
            content="I don't have a function to list all movies currently being shown at the cinema. My available 
tools only allow me to check showtimes for a specific movie when you provide its title.\n\nTo find out what movies 
are playing, you might need to:\n- Check the cinema's website or app directly\n- Visit the cinema's lobby or box 
office\n- Ask me about a specific movie you're interested in, and I can check its showtimes\n\nIs there a 
particular movie you'd like to know the showtimes for?",
            additional_kwargs={},
            response_metadata={
                'model_name': 'poolside/laguna-xs-2.1:free',
                'id': 'gen-1786168375-mBJxHtB7cE4qGCmyM8ZH',
                'created': 1786168375,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--019fdfee-7694-7940-a42d-e0372660e397-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 564,
                'output_tokens': 108,
                'total_tokens': 672,
                'input_token_details': {'cache_creation': 0, 'cache_read': 16},
                'output_token_details': {'reasoning': 0}
            }
        ),
      

# Model Fallback

In [65]:
resilient_agent = create_agent(
    model="openai:gpt-5.5-haiku",     # primary, most capable
    tools=cinebot_tools,
)

In [66]:
result = resilient_agent.invoke( {"messages": [("user", "Summarize my chat? ")]},)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `gpt-5.5-haiku` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

In [67]:
resilient_agent = create_agent(
    model="openai:gpt-5.5-haiku",     # primary, most capable assume its not available or decomissioned, so we fall back to a cheaper model
    tools=cinebot_tools,
    middleware=[
        ModelFallbackMiddleware(
            model_or,   # fallback -- cheaper, still OpenAI, needs no extra setup
            model_groq_lamma70b,   # a further, fully-local last resort -- uncomment if you have
                                    # `pip install langchain-ollama` AND a local Ollama server running.
                                    # Left commented here so this cell runs with nothing beyond
                                    # what Setup already installed.
        ),
    ],
)
print("Fallback chain: gpt-5.5 haiku -> gpt-5-mini.")
print("If the primary model call fails for any reason, this silently tries the next one.")

Fallback chain: gpt-5.5 haiku -> gpt-5-mini.

If the primary model call fails for any reason, this silently tries the next one.

In [68]:
result = resilient_agent.invoke( {"messages": [("user", "Summarize my chat? ")]},)

In [69]:
print(result)

{
    'messages': [
        HumanMessage(
            content='Summarize my chat? ',
            additional_kwargs={},
            response_metadata={},
            id='ba773fa4-4d03-4f36-a7a9-ebaadb2f2878'
        ),
        AIMessage(
            content="I don't have any previous conversation history to summarize. This is the start of our chat, so
there's nothing to summarize yet!\n\nIf you'd like me to summarize something, please share the chat content you'd 
like me to summarize.",
            additional_kwargs={
                'reasoning_content': 'The user is asking me to summarize their chat. However, looking at the 
conversation, the user has only sent one message: "Summarize my chat?" - there\'s no previous chat history to 
summarize. I should let them know that I don\'t have any previous conversation history to summarize.',
                'reasoning_details': [
                    {
                        'type': 'reasoning.text',
                        'format': 'unknown',
                        'index': 0,
                        'text': 'The user is asking me to summarize their chat. However, looking at the 
conversation, the user has only sent one message: "Summarize my chat?" - there\'s no previous chat history to 
summarize. I should let them know that I don\'t have any previous conversation history to summarize.'
                    }
                ]
            },
            response_metadata={
                'model_name': 'inclusionai/ling-3.0-tiny:free',
                'id': 'gen-1786168580-jKmjybHnOodcxapRUc9a',
                'created': 1786168580,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--019fdff1-951d-77c0-8a5f-6d29f68607f9-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 648,
                'output_tokens': 110,
                'total_tokens': 758,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 70}
            }
        )
    ]
}

# Tool Call Limit

In [ ]:
tool_limited_agent = create_agent(
    model=model_or,
    tools=cinebot_tools,
    checkpointer=InMemorySaver(),
    middleware=[
        ToolCallLimitMiddleware(run_limit=8),                              # global, this turn
        ToolCallLimitMiddleware(tool_name="cancel_booking", thread_limit=2),  # tighter, one tool, whole conversation
    ],
)

# 9Aug 
**Fraud detection in fintech:** a `wrap_tool_call`-style hook around every transaction-executing
tool, flagging or blocking based on velocity, amount, or pattern — exactly `CineBotComplianceMiddleware`'s
rate-limiting logic, aimed at fraud instead of booking abuse.

**Healthcare AI assistants:** `PIIMiddleware`-style redaction is often a genuine *legal*
requirement (HIPAA in the US) — the pattern from Part 2's PII sections isn't optional polish in
that industry, it's the difference between compliant and non-compliant software.

**Customer support platforms:** `HumanInTheLoopMiddleware`-style approval gates before any action
with real financial consequence (refunds, account changes) — the exact shape used to guard
`cancel_booking` here, used to guard a real refund tool there.

**Internal developer tools:** audit logging via `wrap_tool_call`, like Part 3's
`BookingAuditMiddleware`, is close to universal in any system where "who did what, when" needs a
real answer — compliance, debugging, or just accountability.

In [47]:
tool_limited_agent = create_agent(
    model=model_or_paid_gpt56_luna_pro,
    tools=cinebot_tools,
    checkpointer=InMemorySaver(),
    middleware=[
        ToolCallLimitMiddleware(run_limit=8),                              # global, this turn
        ToolCallLimitMiddleware(tool_name="cancel_booking", thread_limit=2, run_limit=1),  # tighter, one tool, whole conversation
    ],
)

In [48]:
config = {"configurable": {"thread_id": "tool-limit-demo"}}

In [74]:

for i in range(3):
  result = tool_limited_agent.invoke({"messages": [("user", f"Please cancel my Booking with ID B{100+i} ? ")]}, config=config)
  print(result)

{
    'messages': [
        HumanMessage(
            content='Please cancel my Booking with ID B100 ? ',
            additional_kwargs={},
            response_metadata={},
            id='dedd19a3-279d-48e8-b480-ddd7407f61ac'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'data': 
'gAAAAABqd--Hrx1Osq_wKXMLnhKM2xv8ziPDMEYSDqaKLk1A-0pqMRNkqP8GhM0ztRXqao_We_3IVY8PzKOWpdV4LPJ6tN_VoPNuVLdKd1Z6R9E82b
mY7ZdK5ftUbcCRanidGpdWQQiAMO9edPeHdYM6aJkQ806retx2ja_yKHfMl-HD0JvPPMjBxC6wUe6e2GyRgiIrVlmgOWPo3ccYBfUfFY25G7OWPItNW
v9GwEbtrHk_Z4_Vi7CT2PmhwEHVxHGMeUNO_OWtH6mV0YE90Ee9pSD1mR50AF3xDGrVWyd8YkTpd2O5NTFL4ILEa8IjY6zyB2BXsZdzbdL8siHzH5bD
l7dAr45CP-2d-hXv9xPDDW2_bMbNJwNjpe86l87lyKbp1dp-yZWA8IW7e1SPiBBpZdio7Xh2MyikHD19fsyzM27bYEA38GUxpPW9x24Uu2LS3daMaen
aYnfzvRcRGx_934JmJYBuKKu_2AkYJFFIpjJX4ZdRx-zzjguKC6PPU5bObvU1iMbMgL1szoMtabHVsQQSxaBET65wgf-gTkDNm0dQt6R-NjsiEle40m
CcCCFmwvOCyyQjCH_clYn3VvsjT_Z7nK6FfB9D93ff3-OzuNdV2xbgFp0-Hj8MekGFHoM8KG2bd7uX870OfCEqAy1x96DyGcW9wpfxOeuIxbYsS2VjB
AA8Ic5EcDc23mEP6wJMMemJiUOcwg-aL62WeqOd4uZOyvd2wa6hfCgLoY-Oz2C5GPc7axEtxocnvfKajGSuPrFvFOimb22v1MYIF_6qPFEqpC7nKK4_
1J3wOVYjYJ_9kP9tnNhx8xJ2qU1rci9GSgUOnXFyXuUQJDCAKLYP9dN_z55sTCF4KbtMuXOrliMxWNOAA_Ry0WMwFip6Gd-S-VFPR4a_0JZ2PRfHqhk
M6qqDj8VDL7BqNcWmRjJun9CJ0vhjcndfCgz7_MGvo0nCLjszJA367v1KG57JbU_pFsJo849I7YINnoLO7_AprNgZIfR0oo5EGfrB0H19RKfmQyFUK8
6J.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuYS1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_0c4410397122074f016a77ef87ba74819ea454a38e5a4cc3a5',
                        'index': 0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786244997-vUzMtyl9NuIYTaKwTcRC',
                'created': 1786244997,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0003222,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0001062,
                    'upstream_inference_prompt_cost': 0.000216,
                    'upstream_inference_cost': 0.0003222
                }
            },
            id='lc_run--019fe47f-9bb9-7880-a5f1-87998e7b8521-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B100'},
                    'id': 'call_mlQ9rqDnyQWytf1JH4pUbJup',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 2160,
                'output_tokens': 177,
                'total_tokens': 2337,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 94}
            }
        ),
        ToolMessage(
            content='Booking B100 cancelled.',
            name='cancel_booking',
            id='11f7d178-8833-4f3d-ad65-a414001007ec',
            tool_call_id='call_mlQ9rqDnyQWytf1JH4pUbJup'
        ),
        AIMessage(
            content='Booking **B100** has been cancelled.',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786245000-XFXLCpkdaD24HFVJxq7i',
                'created': 1786245000,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.000284,
                'cost_details': {
                    'upstream_inference_completions_cost': 5.46e-05,
       

{
    'messages': [
        HumanMessage(
            content='Please cancel my Booking with ID B100 ? ',
            additional_kwargs={},
            response_metadata={},
            id='dedd19a3-279d-48e8-b480-ddd7407f61ac'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'data': 
'gAAAAABqd--Hrx1Osq_wKXMLnhKM2xv8ziPDMEYSDqaKLk1A-0pqMRNkqP8GhM0ztRXqao_We_3IVY8PzKOWpdV4LPJ6tN_VoPNuVLdKd1Z6R9E82b
mY7ZdK5ftUbcCRanidGpdWQQiAMO9edPeHdYM6aJkQ806retx2ja_yKHfMl-HD0JvPPMjBxC6wUe6e2GyRgiIrVlmgOWPo3ccYBfUfFY25G7OWPItNW
v9GwEbtrHk_Z4_Vi7CT2PmhwEHVxHGMeUNO_OWtH6mV0YE90Ee9pSD1mR50AF3xDGrVWyd8YkTpd2O5NTFL4ILEa8IjY6zyB2BXsZdzbdL8siHzH5bD
l7dAr45CP-2d-hXv9xPDDW2_bMbNJwNjpe86l87lyKbp1dp-yZWA8IW7e1SPiBBpZdio7Xh2MyikHD19fsyzM27bYEA38GUxpPW9x24Uu2LS3daMaen
aYnfzvRcRGx_934JmJYBuKKu_2AkYJFFIpjJX4ZdRx-zzjguKC6PPU5bObvU1iMbMgL1szoMtabHVsQQSxaBET65wgf-gTkDNm0dQt6R-NjsiEle40m
CcCCFmwvOCyyQjCH_clYn3VvsjT_Z7nK6FfB9D93ff3-OzuNdV2xbgFp0-Hj8MekGFHoM8KG2bd7uX870OfCEqAy1x96DyGcW9wpfxOeuIxbYsS2VjB
AA8Ic5EcDc23mEP6wJMMemJiUOcwg-aL62WeqOd4uZOyvd2wa6hfCgLoY-Oz2C5GPc7axEtxocnvfKajGSuPrFvFOimb22v1MYIF_6qPFEqpC7nKK4_
1J3wOVYjYJ_9kP9tnNhx8xJ2qU1rci9GSgUOnXFyXuUQJDCAKLYP9dN_z55sTCF4KbtMuXOrliMxWNOAA_Ry0WMwFip6Gd-S-VFPR4a_0JZ2PRfHqhk
M6qqDj8VDL7BqNcWmRjJun9CJ0vhjcndfCgz7_MGvo0nCLjszJA367v1KG57JbU_pFsJo849I7YINnoLO7_AprNgZIfR0oo5EGfrB0H19RKfmQyFUK8
6J.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuYS1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_0c4410397122074f016a77ef87ba74819ea454a38e5a4cc3a5',
                        'index': 0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786244997-vUzMtyl9NuIYTaKwTcRC',
                'created': 1786244997,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0003222,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0001062,
                    'upstream_inference_prompt_cost': 0.000216,
                    'upstream_inference_cost': 0.0003222
                }
            },
            id='lc_run--019fe47f-9bb9-7880-a5f1-87998e7b8521-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B100'},
                    'id': 'call_mlQ9rqDnyQWytf1JH4pUbJup',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 2160,
                'output_tokens': 177,
                'total_tokens': 2337,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 94}
            }
        ),
        ToolMessage(
            content='Booking B100 cancelled.',
            name='cancel_booking',
            id='11f7d178-8833-4f3d-ad65-a414001007ec',
            tool_call_id='call_mlQ9rqDnyQWytf1JH4pUbJup'
        ),
        AIMessage(
            content='Booking **B100** has been cancelled.',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786245000-XFXLCpkdaD24HFVJxq7i',
                'created': 1786245000,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.000284,
                'cost_details': {
                    'upstream_inference_completions_cost': 5.46e-05,
       

{
    'messages': [
        HumanMessage(
            content='Please cancel my Booking with ID B100 ? ',
            additional_kwargs={},
            response_metadata={},
            id='dedd19a3-279d-48e8-b480-ddd7407f61ac'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'data': 
'gAAAAABqd--Hrx1Osq_wKXMLnhKM2xv8ziPDMEYSDqaKLk1A-0pqMRNkqP8GhM0ztRXqao_We_3IVY8PzKOWpdV4LPJ6tN_VoPNuVLdKd1Z6R9E82b
mY7ZdK5ftUbcCRanidGpdWQQiAMO9edPeHdYM6aJkQ806retx2ja_yKHfMl-HD0JvPPMjBxC6wUe6e2GyRgiIrVlmgOWPo3ccYBfUfFY25G7OWPItNW
v9GwEbtrHk_Z4_Vi7CT2PmhwEHVxHGMeUNO_OWtH6mV0YE90Ee9pSD1mR50AF3xDGrVWyd8YkTpd2O5NTFL4ILEa8IjY6zyB2BXsZdzbdL8siHzH5bD
l7dAr45CP-2d-hXv9xPDDW2_bMbNJwNjpe86l87lyKbp1dp-yZWA8IW7e1SPiBBpZdio7Xh2MyikHD19fsyzM27bYEA38GUxpPW9x24Uu2LS3daMaen
aYnfzvRcRGx_934JmJYBuKKu_2AkYJFFIpjJX4ZdRx-zzjguKC6PPU5bObvU1iMbMgL1szoMtabHVsQQSxaBET65wgf-gTkDNm0dQt6R-NjsiEle40m
CcCCFmwvOCyyQjCH_clYn3VvsjT_Z7nK6FfB9D93ff3-OzuNdV2xbgFp0-Hj8MekGFHoM8KG2bd7uX870OfCEqAy1x96DyGcW9wpfxOeuIxbYsS2VjB
AA8Ic5EcDc23mEP6wJMMemJiUOcwg-aL62WeqOd4uZOyvd2wa6hfCgLoY-Oz2C5GPc7axEtxocnvfKajGSuPrFvFOimb22v1MYIF_6qPFEqpC7nKK4_
1J3wOVYjYJ_9kP9tnNhx8xJ2qU1rci9GSgUOnXFyXuUQJDCAKLYP9dN_z55sTCF4KbtMuXOrliMxWNOAA_Ry0WMwFip6Gd-S-VFPR4a_0JZ2PRfHqhk
M6qqDj8VDL7BqNcWmRjJun9CJ0vhjcndfCgz7_MGvo0nCLjszJA367v1KG57JbU_pFsJo849I7YINnoLO7_AprNgZIfR0oo5EGfrB0H19RKfmQyFUK8
6J.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuYS1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_0c4410397122074f016a77ef87ba74819ea454a38e5a4cc3a5',
                        'index': 0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786244997-vUzMtyl9NuIYTaKwTcRC',
                'created': 1786244997,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0003222,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0001062,
                    'upstream_inference_prompt_cost': 0.000216,
                    'upstream_inference_cost': 0.0003222
                }
            },
            id='lc_run--019fe47f-9bb9-7880-a5f1-87998e7b8521-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B100'},
                    'id': 'call_mlQ9rqDnyQWytf1JH4pUbJup',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 2160,
                'output_tokens': 177,
                'total_tokens': 2337,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 94}
            }
        ),
        ToolMessage(
            content='Booking B100 cancelled.',
            name='cancel_booking',
            id='11f7d178-8833-4f3d-ad65-a414001007ec',
            tool_call_id='call_mlQ9rqDnyQWytf1JH4pUbJup'
        ),
        AIMessage(
            content='Booking **B100** has been cancelled.',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786245000-XFXLCpkdaD24HFVJxq7i',
                'created': 1786245000,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.000284,
                'cost_details': {
                    'upstream_inference_completions_cost': 5.46e-05,
       

# PII Detection

In [59]:
pii_agent = create_agent(
    model=model_or_paid_gpt56_luna_pro,
    tools=cinebot_tools,
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", 
                      detector=r"\d{4}-\d{4}-\d{4}-\d{4}",  # Custom regex pattern
                      apply_to_input=True),
    ],
)

In [60]:
result = pii_agent.invoke({
    "messages": [("user", "My email is sunj@example.com and my credit card is 3782-8224-6310-1205, can you check showtimes for Dune?")]
})

In [61]:
rprint(result)

{
    'messages': [
        HumanMessage(
            content='My email is [REDACTED_EMAIL] and my credit card is ****-****-****-1205, can you check 
showtimes for Dune?',
            additional_kwargs={},
            response_metadata={},
            id='d5eb9e0e-fe7b-4f00-bf5a-4cdc2551c985'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'data': 
'gAAAAABqgRghud7Zc1DGaEi_zZ4MpWPjAPp0UHjSlXTjGtmfPZApYttycKxeoHDWf-Z96vapKP3uHSLgGeYyv5ONPKhb1sbU65cmcwE-K52AYjLl6-
7_NuVrlLxTSjZroOrQvWFJ-1zRkf7CXS6UUjPph1fWt9GvuLbVM3YQ8Y8IdHvaY0fZh_OngWSAwNd7f3WvArumRilAXil6wk52asWqt5w3m-1AAe44B
4ADSQlhmNCK9XY-5ZaORcdzH2exoMiSst1hX9nqXmx9YVoIvcqmAK3O7YEDvRq8OYWq3P4NdLj7vrZkQ7jfJQfhSMpAradlSxeJaIioFbtbQAaR1-oC
JvJTJE871QT3hBXdedLm1muLPobuSqq8IwILRrVVG9lyNrUqSWo8QWL-YgM0kSR6JEYgsTxAbMR0qO3YTsfvXvjgqRU2HXR3q1F-YY9cBSZ3pdVJbK7
JxEtqhG8IqYV07bTs4w_Gf1jDVzAba4QlTEokBEMq64FiIbeAe2QHZHdXq-xOyptU-2Y_uKgAijht6u6ZjEiv4D1_L7RG_LdgAUzSofR-EnjrvPcCSJ
tkM4jeQAFC0c20HboPz8rb_PYakD9bAWNe7WxSFb0l2O8_er9DD6bfLBHx7jQ-lE6_kYV7nPZwYdYp755PJ-hE-RgY0gVrgNlhFZ5CXUYtGMHNywMfW
uUkuKEm5m2QePGZdjrnM7r2tKtX1KW8zsAxE6jG0EPQZdm1oL_EAfdAewn9RxyqQ4Rc2hZg_EAqth9HDGCDPy8jRfYqUefdgcX6XTrc1ejMUa690HVL
QKnpfZtv9mBzXLsF0HitDBGF6m5lDIwcwbjQa8WbJ7oTeXu2gDoUpjinzKG58ZgKWYQic-rxtxNsb-0ew-BH2TGQvGXjK0-tIAFX97rx9PRgaSGcmMI
nfArSlCwXxh3OxmXld0TdxbSz7HQABYIl6zn4slMt2h7vZNm2TrhUc9EQKrE6nkvkjFlBYHgX0YtMmOHArFaD5hanPXjiiK8RNt4YVcnRRsSa9xGO43
b9YLuRAvxRnaym9NnMcUnbY5-9tK4hnPpYs7ystVRKEZbeARC6Hwe9bLby0WLerskwxlxYGWnTZkztFJ5NFkotQyij7ZhfuOtprtCwVCeTsiJOd-UW0
0wygx6AnEuWadaCJLhqzRY9GEOHXjBH5vTbEV6BNmQ5uOw0Pq4Vqr-Yz2eolzyPnMR7FpYmFTxLj4ZBXJerraJfJFsZGL7wBpyw6Q9chqpKbBXSQXGu
m3b8R-CRMDcxekNxut7HnNlfSHmO2VwGACxVKCm8bxfU5e4gqIi_90kBg2LAwtgk1rOv2xA_CcIAF3t44_snco4MxEgl_V5YB-iKBBLrlPotCdcSKvE
Q8MP06d0SJrxnHbtRjvE_gYpUI7FnekqgyGz4-8fITieTPYaoGLcPuFqo0WF1Bg==.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuY
S1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_04c1621d3b54b416016a811821a44c8190a4593dd31d7bb60e',
                        'index': 0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786845214-6qvCLHN3JfS444pEPqLr',
                'created': 1786845214,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0003445,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0001194,
                    'upstream_inference_prompt_cost': 0.0002251,
                    'upstream_inference_cost': 0.0003445
                }
            },
            id='lc_run--01a00846-319d-7451-8a13-91535d8b2f5d-0',
            tool_calls=[
                {
                    'name': 'check_showtimes',
                    'args': {'movie_title': 'Dune'},
                    'id': 'call_MLvaxa0Moq3I5kDuYb7BVKlC',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 2251,
                'output_tokens': 199,
                'total_tokens': 2450,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 113}
            }
        ),
        ToolMessage(
            content='No showtimes found for that title.',
            name='check_showtimes',
            id='c95645d9-45ad-4400-80ec-f63a9c9aff1c',
            tool_call_id='call_MLvaxa0Moq3I5kDuYb7BVKlC'
        ),
        AIMessage(
            content='No showtimes were found for **Dune**.',
            additional_kwargs={},
      

In [79]:
print(result)

{
    'messages': [
        HumanMessage(
            content='My email is [REDACTED_EMAIL] and my card is 3782-8224-6310-005, can you check showtimes for 
Dune?',
            additional_kwargs={},
            response_metadata={},
            id='c731a601-f724-4cf8-bf94-f60fce231965'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_content': 'User wants showtimes for Dune. We have function check_showtimes. We should 
call it.',
                'reasoning_details': [
                    {
                        'type': 'reasoning.text',
                        'format': 'unknown',
                        'index': 0,
                        'text': 'User wants showtimes for Dune. We have function check_showtimes. We should call 
it.'
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-oss-20b:free',
                'id': 'gen-1786246050-gJd3LoKHDBaYBewjpckQ',
                'created': 1786246050,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 3.29e-06,
                    'upstream_inference_prompt_cost': 4.495e-06,
                    'upstream_inference_cost': 7.785e-06
                }
            },
            id='lc_run--019fe48f-ab3f-7cc2-80e1-b4e7774b1717-0',
            tool_calls=[
                {
                    'name': 'check_showtimes',
                    'args': {'movie_title': 'Dune'},
                    'id': 'call_6170717EDE424651ACE35248',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 310,
                'output_tokens': 47,
                'total_tokens': 357,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 20}
            }
        ),
        ToolMessage(
            content='No showtimes found for that title.',
            name='check_showtimes',
            id='245ab4a8-5c65-4d57-a4ce-6727171565d8',
            tool_call_id='call_6170717EDE424651ACE35248'
        ),
        AIMessage(
            content='I checked the showtimes for Dune, but unfortunately no showtimes were found for that title in 
the current system. This could mean:\n\n1. Dune is not currently showing at any cinema in the system\n2. The movie 
title might need to be specified differently\n3. There might be a temporary issue with the showtime data\n\nWould 
you like me to try searching for a different movie title, or would you prefer to check what movies are currently 
available?',
            additional_kwargs={
                'reasoning_content': 'The function returned "No showtimes found for that title." This means there 
are no showtimes available for "Dune" in the system. I should inform the user about this result.',
                'reasoning_details': [
                    {
                        'type': 'reasoning.text',
                        'format': 'unknown',
                        'index': 0,
                        'text': 'The function returned "No showtimes found for that title." This means there are no
showtimes available for "Dune" in the system. I should inform the user about this result.'
                    }
                ]
            },
            response_metadata={
                'model_name': 'cohere/north-mini-code:free',
                'id': 'gen-1786246056-qbhC3Ho8064JgdIk4fDW',
                'created': 1786246056,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'c

In [80]:
print(result['messages'][-1].content)

I checked the showtimes for Dune, but unfortunately no showtimes were found for that title in the current system. 
This could mean:

1. Dune is not currently showing at any cinema in the system
2. The movie title might need to be specified differently
3. There might be a temporary issue with the showtime data

Would you like me to try searching for a different movie title, or would you prefer to check what movies are 
currently available?

### https://reference.langchain.com/python/langchain/agents/middleware/pii/PIIMiddleware

In [81]:
import re

In [82]:

def detect_booking_code(content: str) -> list[dict]:
    """Detect CineBot's own booking code format: BK followed by 4 digits."""
    matches = []
    for match in re.finditer(r"BK\d{4}", content):
        matches.append({"text": match.group(0), "start": match.start(), "end": match.end()})
    return matches


In [83]:
custom_pii_agent = create_agent(
    model=model_groq_lamma70b,
    tools=cinebot_tools,
    middleware=[PIIMiddleware("booking_code", detector=detect_booking_code, strategy="hash")],
)

In [84]:
result = custom_pii_agent.invoke({
    "messages": [("user", "Can you check the status of my booking BK1044 for me?")]
})

print(result)

{
    'messages': [
        HumanMessage(
            content='Can you check the status of my booking <booking_code_hash:94be15f9> for me?',
            additional_kwargs={},
            response_metadata={},
            id='5f68cae4-ac1d-4f00-b367-363f42319e67'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'tool_calls': [
                    {
                        'id': 'zqtj8wrss',
                        'function': {'arguments': '{"booking_id":"94be15f9"}', 'name': 'check_order_status'},
                        'type': 'function'
                    }
                ]
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 21,
                    'prompt_tokens': 582,
                    'total_tokens': 603,
                    'completion_time': 0.043088173,
                    'completion_tokens_details': None,
                    'prompt_time': 0.029376434,
                    'prompt_tokens_details': None,
                    'queue_time': 0.058468085,
                    'total_time': 0.072464607
                },
                'model_name': 'llama-3.3-70b-versatile',
                'system_fingerprint': 'fp_dae98b5ecb',
                'service_tier': 'on_demand',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'groq'
            },
            id='lc_run--019fe49b-06f0-7cd0-8164-b960ecc53712-0',
            tool_calls=[
                {
                    'name': 'check_order_status',
                    'args': {'booking_id': '94be15f9'},
                    'id': 'zqtj8wrss',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={'input_tokens': 582, 'output_tokens': 21, 'total_tokens': 603}
        ),
        ToolMessage(
            content='Booking 94be15f9: confirmed, 2 seats, Interstellar, 7:00 PM.',
            name='check_order_status',
            id='c8dfacf6-bb85-428b-b584-2add1494faa8',
            tool_call_id='zqtj8wrss'
        ),
        AIMessage(
            content='Your booking for the movie "Interstellar" at 7:00 PM is confirmed, and you have booked 2 
seats.',
            additional_kwargs={},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 27,
                    'prompt_tokens': 636,
                    'total_tokens': 663,
                    'completion_time': 0.082551602,
                    'completion_tokens_details': None,
                    'prompt_time': 0.032291193,
                    'prompt_tokens_details': None,
                    'queue_time': 0.056338445,
                    'total_time': 0.114842795
                },
                'model_name': 'llama-3.3-70b-versatile',
                'system_fingerprint': 'fp_dae98b5ecb',
                'service_tier': 'on_demand',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'groq'
            },
            id='lc_run--019fe49b-07c1-7be2-b7d7-22cbd98935fa-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={'input_tokens': 636, 'output_tokens': 27, 'total_tokens': 663}
        )
    ]
}

# TO Do List

In [89]:
todo_agent = create_agent(
    model=model_or_paid_gpt56_luna_pro,
    tools=cinebot_tools,
    middleware=[TodoListMiddleware()],
    system_prompt=
    """You are a helpful assistant that can manage a to-do list for the user."""
    
)

In [90]:
result = todo_agent.invoke({
    "messages": [("user", "I want to plan a movie night: check what's showing, pick something good, and book 2 seats.")]
})

In [91]:
print(result)

{
    'messages': [
        HumanMessage(
            content="I want to plan a movie night: check what's showing, pick something good, and book 2 seats.",
            additional_kwargs={},
            response_metadata={},
            id='7a68f975-0515-4533-871f-60fa1faca4d0'
        ),
        AIMessage(
            content='What genre or type of movie would you like—comedy, action, thriller, drama, animation, or 
something else? I can then check showtimes, recommend one, and book 2 seats.',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'data': 
'gAAAAABqd_wEcIje25xkcbIVv6bxIDKsVHBoaixFosPIl1CmXmUoU3ccwZ14Aebjbn6i3gX0qX6w2ETx0XvCDqrplftzzLbmNqAvqSTEwSsuUreOxm
qIXHD_Rq2zWvAdkH5MXTYtMvz-2veryBMsatkqsY0S50rNCwegnTsXnrKII9OodLXfz_Lhp9vb2x_7Aae65gu1cuVVr6bK_W_VPiKHB8Zbk4H7I7lR9
KZcbxF9Fj59lY0zxpUFtpYg7qpfawDhAT5w4_buvi-vyiwqq0U-mhr62Z-33OLFs36zt22RxTuSY7joWJdajYLkauk6sxSLlz3x6VblNN59IGiSBiHG
KdEtCv1ItUce7DWjlnX1JqpTf82OKaVYNquyI1D8qLkEynyjQr3_7bA3-xG7lBT7yWshrSxcpkOfb5nR4TgWuCpoTHOVBaE1QrFBwSupLxESvP1yR5d
xPXEy9xyGrWdFA2R34HxH7G66pd_WhrZi-qE8Uq9_76iPbKKJIT-M1zAmi1wrbdcYuNxe4KLCGM23q5Luqph9_3TisvHAQuRhleAKKbE4GsJpzWzM3C
2JVS2Y_Lkt0uKuyzox29R0Z_7ASpxsA9UtmhQfYTFhV3-3wI1tC9CkQI8vurcb58a2X2G8V-TRJ--xdf6vCZZZ_873Y27TLu3nVWp_SMrdrNLFxQYja
8afgUXF2C2tWKjZZCCJ-Hx_V5ZjoIWgiHm9j5UcJmGJ0_-lq68UnEmS4QGF35LK1vqjNcEtJ6e_ijNUPkp4exkOB1uB3JdXHP5OSnyim2LS8zSDScpR
g2fBrH8G-M7L12mWV378S9t1qLndj1YEvqlAqae5k-HUP23nXZG2ZQf0KTc2CzbJzB49F6oSH4p_EDfX6RzJEVIi5vE_EAq3RNK51QKrB6gCZ0txPY8
ZynYn6WF9_0WvPdmwlQwS-sRLqZ0Av49XZBHlvP3D6Sb6cBTtZmh-jp8UPmooqmzXwhCcwxsPbO-twOOdgt7eOFNH7vXign6fH6DgVQewxApULCa97Q
QcGl2pi-8wZuL-H0bcdrJ7ktWaJ5saZSbJQChlbNLOxmz0YIUg0P3bnJugPL-O4MKR9geU1_SVgbKkJ8OZT0QElw4MfC013mpSoHNLjNL4p66Gt1FO_
Zs7rtAszVBmPqQavfmaWm5hhn7KQSIL5K-hC5HBe98SGIpgqv6u4iIBEHORLF5i9TSrU3Q-byNBck9kzY3C0Cr78tGiSh2mdPIjTxrVjTL3CXkmM2iN
mEOQh_joT2NXo5ECRp9EDhxgZIrB3CDZZaezSFg6iJOVsjYbz3waOhdOeejErcAAMxqUaE3ggj0seI6H84pwpbRNMUT3g1WrZKw3msH40JHm0uFA-AX
Q_aVv5TM4mYgLYRFyWj8QpL7otZCyWKWlpwGoA7Z6xEzaOL76B5qBP8dXulSIQuiGLEYinbL6b7y89HpoiJXSrNt7ZLmmF9VU8_cEOG7WEeVnQNJtql
JfwCr5SAaeQTTjEHb7BH8jBCCTF2ccnbdIMtRiDuUMvk3t6C3X0bKZI0fI9euvsCcfEhAV_6ae6dK2DqcQ2rS2xsNW2O6mLzPkeWZxVNcV8Qr4nfPsx
R0WEBmQ48pxeUvF6qVNJFXMdWjrgRCyjpQqvIUoaOuzVLj72EOkM7vac2t7wWr-3gNlKVKOTGvBguAODiXIPkIfW84a6o0PlsJjb4n7EYk3T-CvJcxK
thHNWebUrfJnBDZKsyevgVv1YWcQm95yDtlral4lqjhRHgsIFegl25PYW0oCKZHKx8Nu4FZax7_XSQpUOC0vJBw9CQR4-UT9fRZaMWsFkh1kpE3gcHT
oNzfY1f8=.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuYS1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_049fc4cb35e5453d016a77fc0414c881a28cd2387d0dbb0c25',
                        'index': 0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786248191-lkS7LZZWm2hVdZonIVwr',
                'created': 1786248191,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.00067312,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0002922,
                    'upstream_inference_prompt_cost': 0.00038092,
                    'upstream_inference_cost': 0.00067312
                }
            },
            id='lc_run--019fe4b0-5788-7463-acfb-26f2de50be98-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 6358,
                'output_tokens': 487,
                'total_tokens': 6845,
                'input_token_details': {'cache_read': 2832, 'cache_creation': 0},
                'output_token_details': {'reasoning': 378}
            }
        )
    ]
}



# LLM Tool Selector

In [92]:
for tool in cinebot_tools:
  print (tool.name)

check_showtimes

book_seats

cancel_booking

check_order_status

get_refund_policy

lookup_seat_map

### to check which tool is being selected by the  LLM Tool Selector middleware

In [93]:
from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def show_tools(request, handler):
    print("\nTOOLS SENT TO MODEL:")
    print([tool.name for tool in request.tools])

    return handler(request)

In [94]:
selector_agent = create_agent(
    model=model_or_paid_gpt56_luna_pro,
    tools=cinebot_tools,
    middleware=[
        LLMToolSelectorMiddleware(
            model=model_or,     # can be a CHEAPER model than the main agent
            max_tools=2,
            always_include=["check_showtimes"],  # always kept, doesn't count against max_tools
        ),
        show_tools
    ],
)

In [95]:
result = selector_agent.invoke({"messages": [("user", "Can you cancel my booking with ID B1234?")]})

TOOLS SENT TO MODEL:

['cancel_booking', 'check_showtimes']

TOOLS SENT TO MODEL:

['cancel_booking', 'check_showtimes']

In [96]:
print(result)

{
    'messages': [
        HumanMessage(
            content='Can you cancel my booking with ID B1234?',
            additional_kwargs={},
            response_metadata={},
            id='fc65f418-e682-419a-90f8-13d7069d95c2'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'data': 
'gAAAAABqeAJIUug1THuOgKKh0fcS0zB5LifUf5ggFYQS7wUcNVi0nTCfu71RQ7v3WowvKHnY5diM1MpuboMP3YsZC-_YGxtvG0kIM5ZqQW1-aAT3iT
XXcM7mwHh2ZzdNKWmYi2ICnxcAWJE40taJUPULWKItuiyTNWvx2oVRKeVoPyW7q3IAkBfFPysMI4eX_jcJQFaspzLKBVoLLjg0OJkB7OY9EnXN3fz2M
2vWwqmSvTJ42OZ2f7cFqd62pX-kJ-eQf3EWUIYkxHN4aeD0jCbvK50Y4Kj3fQqTunn2fsc05LLFDXw_3uGAYkrePuSablXaIt7RJzUSWON1j-dInl4H
ZQviQBE8Jxo_9twL_fcDZ0bnkEDzKojLcvL35-evozp9qf4cXVdZ-uh3tSzRCZMQCGYABva3Q4U2q_f7Slonu9k3e9NyCkOkLwlWHeglU7nymdOB9eD
eZ3LyINIJhc31UQGEAA96m4TIY8bCs3fAkcbfD_mxmzGEyQiKIDzEcujJ65aV1KvQVybxXHv3pcoSL0Jcn9xH80GPB-_uztxoi4xlFfSlVdl4Vuni6Y
oF8TKXfhiym1vZOTetVvxevLkHhtKdqXwC0m8P6lTcUfI9W3vnqcbueFbkoJO0pqurIdIFswY_aZx2csS63JMo-Q-wa2JxBWWh8Q8GTOLryYUUIA4x2
LixRpTc8fYKStUcj6hF2QHiLKX5jopuPfc8dXDQT_7FOinWcu2DJBNUUcXcaJwW4kQoX-D_cyFm2JccOHyXRKZDPNnyGzaLFLm9jpGyCjsnCYHU7TRe
tnf7L7EYtiWbek1UehLRLk6ff5EniRwx85eS98MUO77QpeCOfZy5m9rQYFtwWja7R4jnO8l0XmwX_RDlEF3hG1CTzZLLr2inm9Rs9A7rvLQWMywqCMP
k5zPzWliRbqkHQwloqDF-jCjqbDaGhV5vxIZJ1GjcWCLOagbWaPUYsF8XCXtw1k-1kkt88b9430AIeP347CjY1VVXzZITHaY=.eyJlbmRwb2ludF9zb
HVnIjoib3BlbmFpL2dwdC01LjYtbHVuYS1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_08d43ddc51619f4d016a780248e58c81a191129a6f23abd2f5',
                        'index': 0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786249798-iC2SmlBC0mAnFf2Go1J5',
                'created': 1786249798,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0002904,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0001104,
                    'upstream_inference_prompt_cost': 0.00018,
                    'upstream_inference_cost': 0.0002904
                }
            },
            id='lc_run--019fe4c8-df72-71d3-843f-37eb17c860c8-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B1234'},
                    'id': 'call_M11XqAlYrd9Aq4X8UXfYT6DZ',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1800,
                'output_tokens': 184,
                'total_tokens': 1984,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 98}
            }
        ),
        ToolMessage(
            content='Booking B1234 cancelled.',
            name='cancel_booking',
            id='a1cbacf2-7f20-43f7-8904-4a87bf2425d6',
            tool_call_id='call_M11XqAlYrd9Aq4X8UXfYT6DZ'
        ),
        AIMessage(
            content='Your booking **B1234** has been cancelled.',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786249805-SZ97IMOn7OZVLXD6AVqX',
                'created': 1786249805,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0002519,
                'cost_details': {
                    'upstream_inference_completions_cost': 5.82e-05,
                   

In [97]:
result = selector_agent.invoke({"messages": [("user", "Can you provide information on refund status")]})

TOOLS SENT TO MODEL:

['get_refund_policy', 'check_showtimes']

TOOLS SENT TO MODEL:

['check_order_status', 'get_refund_policy', 'check_showtimes']

In [99]:
print(result)

{
    'messages': [
        HumanMessage(
            content='Can you provide information on refund status',
            additional_kwargs={},
            response_metadata={},
            id='23ef4068-5fd4-439d-9b18-dcebb1a6c14d'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_content': '**Clarifying refund status**\n\nI need to clarify what the user means by 
“refund status.” They might be looking for specific details that I can’t access right now. I could provide general 
policy information, but that might not be what they need. I should consider asking for their booking details to get
more context. Since I have access to the refund policy tool, I’ll think about how to best retrieve that relevant 
information.',
                'reasoning_details': [
                    {
                        'summary': '**Clarifying refund status**\n\nI need to clarify what the user means by 
“refund status.” They might be looking for specific details that I can’t access right now. I could provide general 
policy information, but that might not be what they need. I should consider asking for their booking details to get
more context. Since I have access to the refund policy tool, I’ll think about how to best retrieve that relevant 
information.',
                        'type': 'reasoning.summary',
                        'format': 'openai-responses-v1',
                        'index': 0
                    },
                    {
                        'data': 
'gAAAAABqeAKLp9azmC8ItykiZmYELehTtbBCXWK8PP18BmVapypYekXIlOf28eUkuCdbySWGe8ETEBJQQhw7bdBcxxhXTzn8GKUibGP1VeL0sSYGK0
GSUrIyKTdo9G49nGa3kGT3wUx-9Gk5B5N4NI6JqG_YyJcd90LOUHC5-cUJ2_PWVruUXD8sgWjnqBjf6jLEXNaNWAOx0vcm_qqH7gwIurZ11vw5SHG9H
DMh05RxYtKJP97-_RbfCzCsh0bs3PAy5MgaxCxLX_tYiOI0rJ8BqmADOW-T96IHuLFPbNt4f1MZxwVFIOe-S1nxL7HWQho5UoeAzPk_ngj3VHkkSqdl
vU_ysb3lkrhzfA7kCICOXD0L2Pjj0Rl05lpyyDQauFWPDBYjEJ_ALZxA7gkp45nnzVOEP6GB43bhSrI5Iq3W1WzwaJrqd-34nZRmSPr0gS6wQqB4R7L
6_yDh_alJU2VJ5QxQ1AfpOeb7CfNBWPojmNvHJdwiu-oEql4_mHnlRVoFEKsHsFn6d6LoZE7Wq-WablYajyqZs1djvXX4T5g0CcJcItvc0T8UrXyGba
dfbjqb5MubshPF15Dv9cozbs_jb2ccSt8qY2qSvFSo1wZvscyUFlUwWlezQ9sAQBz1c3KvTBL8X8bOj5glUc7eJowYQhPL5TN_gFUGfTFwKGPMVSat8
O-BXf6nBO3kNSFBwkMnMdNTlquwnlsLp3FMDV_hN9zFdTKeCvlOdcfC-PKdF0DS54GJ2VCicFTUsQZOOJTP1KYCKEe50gaN7CY8TsEwcjZSP9TmYZWX
x4OKzctmazs58FAfs2Zi_9yWKkUa4L32-gvkUWSU9eXJp_GZyYMm9tuR3RklMM0drKrI9-g_tOmq-NkywbpRZ7_l_ifNPtNl4TgR5jFvSvB_sjA2T_O
Sdgca9rT9UZ1oo0lXj9LeIr97IAwwyc5n-oqu_ox9xbeHhxRe9krf1yDCGRjATtoeB70cFIQhKEf97NuAsbSVYViA7sSFlYNwxWaQA0z2uiJt_mkXyC
ia8cABNDr_uRRLzob83DIIrrjAVK5hFeNnlwnMMehtcPA74PQ3NDgjLYPLH8HwSYCS0HNRhj6VoGF4Qnm7D82mPUr2S7beKf3fpCMu-eD3DMPHSWvZv
tmq2t6UzGj8GT2qv9_D-Jx4qRjEVT9uZNXU0WzHHOusJpd24eDWYZogUPM_Sc95JLdRBzq1Y7OfdsYpT1S20dEl0ZzRQ-NJ3LsaNcULT0iohEJsjfQS
nlUcZo00vFh4qAPfzCRtcRFCDW-3cx2pYvJIcwl6yiJ6XhOKhiiY4AeTPbvnwhgoOv5cYHA=.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01Lj
YtbHVuYS1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_0d0f8ef066b31fee016a78028b4f088191b95bc0a7e3b0f6cb',
                        'index': 1
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786249864-Ljv3QAbM6IrPnxfIMsq5',
                'created': 1786249864,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0003163,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0001314,
                    'upstream_inference_prompt_cost': 0.0001849,
                    'upstream_inference_cost': 0.0003163
                }
            },
            id='lc_run--019fe4c9-e22b-7353-9057-422f473ba494-0',
           

# Tool Error

In [105]:
import langchain

print(langchain.__version__)

1.3.13

In [3]:
from langchain.agents.middleware import ToolErrorMiddleware

In [21]:
@tool
def lookup_seat_map(movie_title: str, seat_number: str) -> str:
    """Look up a specific seat -- fails if the seat number format is wrong."""
    if not seat_number or not seat_number[0].isalpha():
        raise ValueError(f"Malformed seat number '{seat_number}' -- expected a letter+number like 'A12'.")
    return f"Seat {seat_number} for {movie_title}: available."

In [22]:
cinebot_tools = [check_showtimes, book_seats, cancel_booking, check_order_status, get_refund_policy, lookup_seat_map]

In [24]:
def on_seat_error(exc: Exception, request : ToolRuntime) -> str | None:
    if isinstance(exc, ValueError):
        # Return the EXCEPTION TYPE, not str(exc) -- internal detail never reaches the model
        return f"`{request.tool_call['name']}` failed with {type(exc).__name__}. Please provide a valid seat number like 'A12'."
    return None  # anything else propagates and halts the run


In [25]:
error_handled_agent = create_agent(
    model=model_or_paid_gpt56_luna_pro,
    tools=cinebot_tools,
    # middleware=[ToolErrorMiddleware(on_error=on_seat_error)],
)

In [26]:
result = error_handled_agent.invoke({"messages": [("user", "Look up seat 12 for Dune Part Two")]})

ValueError: Malformed seat number '12' -- expected a letter+number like 'A12'.

In [30]:
error_handled_agent = create_agent(
    model=model_or_paid_gpt56_luna_pro,
    tools=cinebot_tools,
    middleware=[ToolErrorMiddleware(on_error=on_seat_error)],
)

In [28]:
from rich import print

In [31]:
result = error_handled_agent.invoke({"messages": [("user", "Look up seat 12 for Dune Part Two")]})
print(result)

{
    'messages': [
        HumanMessage(
            content='Look up seat 12 for Dune Part Two',
            additional_kwargs={},
            response_metadata={},
            id='675aa6bc-d5d4-4613-9421-9165bcb404d6'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'data': 
'gAAAAABqeCWdmQztF2EgbfOqOQ-ytL8Vie8-_NkLs2ioRNIM2mfXWopkg7Vzy15GYoZsAstT-J60iHXMHUhIw72WnovAnPkj6rllokYyykT4p8aTK_
dUq_HXb-rDKj87tMXZYifiGQddZMCjCFcmecG7s0d1nGk0xcVLiAKQ8ok2IMcLxEcS3bCRalqBO78D0242pXgvnhEXY503-lY3Yc3Pq1YO6hRfln-uM
BiKpx6lrqA5WZwdEGCesgKDxgZ-p7JXjR1GmKcxlEYeeVKx7P2GV-PouyynBf4c2W9KbV_vIShWaXjn_6gLsMULwDIu5ER0cbZLDGZal-e_cwsq63ui
zciKqM7xM8tJfw07T3AIunZ4UWrIF3tBBZ0D9nagN6nD61D8cKF5vskanYHvDyhy2IF_XXM8r_0C4LBo6bKuDEVatkJzuCiaw7Hfj8MwsHWrDNXWny7
dZMWZuEJDlgFaiGK6eZVYJf84i2mUoay8b93EdzenkV9Px5BsEqsQYFJJ0Jwyo-4wwFVWyqh_hxpOoMVvgrnKRCwnQjX-JoBUOMXmW3wTZJIeP6XQqa
7WXTz1IvJ4vs6MW60lvlqI_ghMDBaa_4c8GBjYQNwYNrr_8PVntF17kOTjqKqKZXOWArrniSoW38LTq0PLNIiA30-lKRRGUq8D16SK8Tc_zJYP3q4Ki
Ks9cPWe3KsoijYWgso4WT_ZqXUfMuP2rmchw1uPyBZy4T-NYfL039bgz5f5SJ7h8WXmfhD3HPd8guktg6ae2vmaD4UZqnOU2Tc5j6sgdcWKXbV2kgxa
WmPcc6qSf1YPp4YF1kFXXDs943W1zb3lLx9Z3JJ3bLcs5uRltEPTRTy-VxjCmMu0NyIb3lWU_isU9Z5DfZQV8vyu445z5xIYWk-l_wDV9s-ST0X37mP
Rd4A416lmYsvr9zLxcAxsNzcsB-xlY_SW9CTkOSAqXpXTviqUUWOI.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuYS1wcm8tMjAyN
jA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_0e46d38fe7bd8302016a78259da51481a392edefb35f77d09f',
                        'index': 0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786258843-y91Pp2D31umyRu23LDTy',
                'created': 1786258843,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0003297,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0001146,
                    'upstream_inference_prompt_cost': 0.0002151,
                    'upstream_inference_cost': 0.0003297
                }
            },
            id='lc_run--019fe552-dd16-7233-a041-223e7a954d87-0',
            tool_calls=[
                {
                    'name': 'lookup_seat_map',
                    'args': {'movie_title': 'Dune Part Two', 'seat_number': '12'},
                    'id': 'call_Oa8nQXQJ0xCnUAV9dD5LN7DC',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 2151,
                'output_tokens': 191,
                'total_tokens': 2342,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 83}
            }
        ),
        ToolMessage(
            content="`lookup_seat_map` failed with ValueError. Please provide a valid seat number like 'A12'.",
            name='lookup_seat_map',
            id='19c07696-a387-421a-a91d-a6c2f4e98e09',
            tool_call_id='call_Oa8nQXQJ0xCnUAV9dD5LN7DC',
            status='error'
        ),
        AIMessage(
            content='Seat numbers require a row letter, such as **A12**. Please specify the row for seat 12.',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786258846-5F55g16coyGvelsmsMJp',
                'created': 1786258846,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0003426,
  

# Tool Rety

In [32]:
from langchain.agents.middleware import ToolRetryMiddleware

import random
random.random()

0.025777375955785664

In [33]:
import random
@tool
def flaky_showtime_check(movie_title: str) -> str:
    """Check showtimes via an external service that can transiently fail."""
    if not random.random() > 1:
        print("Facing Connection Error")
        raise ConnectionError("Simulated network failure -- exactly what a real external call risks.")
    return f"{movie_title}: showing at 8:00 PM."


In [34]:
resilient_tool_agent = create_agent(
    model=model_or,
    tools=[flaky_showtime_check],
    middleware=[
        ToolRetryMiddleware(max_retries=3, backoff_factor=2.0, initial_delay=1.0, on_failure="continue"),
    ],
)

In [35]:
result = resilient_tool_agent.invoke({"messages": [("user", "Check showtimes for Interstellar")]})

Facing Connection Error

Facing Connection Error

Facing Connection Error

Facing Connection Error

In [36]:
print(result)

{
    'messages': [
        HumanMessage(
            content='Check showtimes for Interstellar',
            additional_kwargs={},
            response_metadata={},
            id='1ec8a013-41b4-47b0-a322-63c16e3d0735'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'poolside/laguna-s-2.1:free',
                'id': 'gen-1786259004-jznSmYBuL4ul3eRxmDxt',
                'created': 1786259004,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--019fe555-538f-7ab3-8b2c-4ff4649829b2-0',
            tool_calls=[
                {
                    'name': 'flaky_showtime_check',
                    'args': {'movie_title': 'Interstellar'},
                    'id': 'chatcmpl-tool-abe862411f7c456a84aa042196740d4a',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 151,
                'output_tokens': 27,
                'total_tokens': 178,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Tool 'flaky_showtime_check' failed after 4 attempts with ConnectionError: Simulated network 
failure -- exactly what a real external call risks.. Please try again.",
            name='flaky_showtime_check',
            id='0da62325-f0be-4374-943e-68f78fbd6f6e',
            tool_call_id='chatcmpl-tool-abe862411f7c456a84aa042196740d4a',
            status='error'
        ),
        AIMessage(
            content='I apologize, but the showtime check for "Interstellar" failed due to a simulated network 
failure. The service appears to be experiencing temporary connectivity issues. \n\nSince this is a known flaky 
service that can transiently fail, I\'d recommend trying again in a moment. The system will automatically retry a 
few times before giving up, so please attempt the request again and hopefully it will succeed on a subsequent 
try.',
            additional_kwargs={
                'reasoning_content': 'The function call failed with a ConnectionError after 4 attempts. This 
appears to be a simulated network failure as mentioned in the error message. The function is described as "flaky" 
and can transiently fail, which is exactly what\'s happening here.\n\nSince this is a simulated failure and the 
function is designed to be flaky, I should inform the user about the issue and suggest they try again. The function
might work on a subsequent attempt.',
                'reasoning_details': [
                    {
                        'type': 'reasoning.text',
                        'format': 'unknown',
                        'index': 0,
                        'text': 'The function call failed with a ConnectionError after 4 attempts. This appears to 
be a simulated network failure as mentioned in the error message. The function is described as "flaky" and can 
transiently fail, which is exactly what\'s happening here.\n\nSince this is a simulated failure and the function is
designed to be flaky, I should inform the user about the issue and suggest they try again. The function might work 
on a subsequent attempt.'
                    }
                ]
            },
            response_metadata={
                'model_name': 'cohere/north-mini-code:free',
                'id': 'gen-1786259025-YQbkiwQufa7hzgSNXQ9o',
                'created': 1786259025,
        

In [37]:
initial_delay=1.0
backoff_factor=2.0

In [42]:
retry_number=2

In [43]:
delay = initial_delay * backoff_factor**retry_number
print(f"Retry {retry_number} will wait {delay} seconds before retrying.")

Retry 2 will wait 4.0 seconds before retrying.

In [44]:
import time
import random

last_called = None


@tool
def flaky_showtime_check_time(movie_title: str) -> str:
    """Check showtimes via an external service that can transiently fail."""
    global last_called

    current_time = time.time()

    print(f"Current time: {current_time}")

    if last_called is not None:
        print(
            f"Time since last call: "
            f"{current_time - last_called:.2f} seconds"
        )
    else:
        print("First call")

    last_called = current_time

    if not random.random() > 1:
        print("Facing Connection Error")
        raise ConnectionError("Simulated network failure")

    return f"{movie_title}: showing at 8:00 PM."

In [45]:

resilient_tool_agent = create_agent(
    model=model_or_paid_gpt56_luna_pro,
    tools=[flaky_showtime_check_time],
    middleware=[
        ToolRetryMiddleware(max_retries=3, backoff_factor=2.0, initial_delay=1.0, on_failure="continue"),
    ],

)

In [46]:
result = resilient_tool_agent.invoke({"messages": [("user", "Check showtimes for Interstellar")]})
print(result)

Current time: 1786261943.745409

First call

Facing Connection Error

Current time: 1786261944.576501

Time since last call: 0.83 seconds

...

Facing Connection Error

Current time: 1786261960.9261525

Time since last call: 4.38 seconds

Facing Connection Error

{
    'messages': [
        HumanMessage(
            content='Check showtimes for Interstellar',
            additional_kwargs={},
            response_metadata={},
            id='861518ce-278c-48ad-b697-c936ceaaf8fd'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'data': 
'gAAAAABqeDG4eFe8sDzYMnCddsJu7jhy6a3IFwy12x2t22qnW4SDUIQTfGDssGnhKx0QQ7NTBUaFLH08A3nznETWKxVIy69RlaCqFyxVEYGIwPGd-e
XFpMstNQhGpoH8G8O4xuR4hN9Q0Svi9NEi-ecs9Bt6pi0FEIRjMBt2GiQMGTROTw8qglIpLjLt6SikN0_osuvmgnfzycme7pgyXKLxkfY-XPcdgdi91
GuU_Xwt_SAEPJIBKQYLmPF3oGHZ-rrzPy6tAkD8YtnbahkWFbpfLPm1lmDZZfZEoYAgVcABtpDZNEJ_a7_Yti8PArnYcNtQJsqvLQ0YGi7R1qNQN1UO
JDG958Cl2OEVNZ1Q_Ci62vp6TSdk3EawDsAm-raj5ZuFEEzMNHFfXusqMsEEVONCz0ZgpOtB_5J5XFRKb_TIvcJq8tUvz1bgefvNsYPYetoDrcj4i_4
-U1p7zqct1eF_76iOYATsRegXnKy0vtugr67uRwRRi_QwXEyNUnwMCEaP91hg__snA1OME7wBqxMHIgWEmnQXGvctz0H3x5BTiP8h9Vf3sILfO3gzEE
ztPp2LNnjOHag_TOE8Ua7vRFXaGL9Ocn0kiFC7t3G3aN6jdky9EEZaJVk0yufr2Vtq6oLYjL3Kba5K30XytiZUF7MCBujRGtcNmazflv69RKJ7kysFp
xPxibXdwegoTDDcNuzO-bD01he07ScI9vmDJ_Cjd2-OyItBqP9ftvyGCcNDPvLsFHLO7-9v6CslV5vIC70YYz52qGOqz874MMv4m6aNJmAz82C5M0AE
ZBBnAuATMsYgLNPQjva5O5gC-0LRFe3EJlwCC8jASRL2NTpJrzYooXUwzmt3nQCXn7zcvEzSZA-EBhilbMQjCJKKmZc2BKSADFuHGgsbGOFNwQQeMEN
x0v7AzFYORCFOktyfALr2HV0gtmiRfPCXwWaKPuH1V5lacjUrdjgX.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dwdC01LjYtbHVuYS1wcm8tMjAyN
jA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_01d7e59cf1c72b28016a7831b8f0448193b66064aa5ca75e84',
                        'index': 0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786261942-G3VhlqwihguKtZkgQHV4',
                'created': 1786261942,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0002779,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.000108,
                    'upstream_inference_prompt_cost': 0.0001699,
                    'upstream_inference_cost': 0.0002779
                }
            },
            id='lc_run--019fe582-2916-7223-ac90-55f48926eecb-0',
            tool_calls=[
                {
                    'name': 'flaky_showtime_check_time',
                    'args': {'movie_title': 'Interstellar'},
                    'id': 'call_e4d3p4O4R2c736rcPD8m9ymp',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1699,
                'output_tokens': 180,
                'total_tokens': 1879,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 85}
            }
        ),
        ToolMessage(
            content="Tool 'flaky_showtime_check_time' failed after 4 attempts with ConnectionError: Simulated 
network failure. Please try again.",
            name='flaky_showtime_check_time',
            id='2d7ffa7b-ced3-4468-b610-97c3347b25c1',
            tool_call_id='call_e4d3p4O4R2c736rcPD8m9ymp',
            status='error'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'data': 
'gAAAAABqeDHCJsDRE6YyRk3cR3utvycDgD71yQSCNOXA_hTOkGYB7w3zpqt0h1s6xj45IKcsYOzrHSHG_YjfU0fD-BK_p1hr6VWjQ-bH-S2DvIO8Z_
xyMg-TuwoNLtMDD8wAR_Ave9tS9lj64sXw-cYa5SdqixEe9UButj6E0m9LQl-nL9uMdOR82N-Z7T9sw3GQNpPHMjTicl-3hmyeelpMyjkMkzbf1zvoB
Mva-kh3geLWwJgJEXUYUJ0dm_MFD55WcmQxsCH2HGFlwGRuZ1j0251GnYQcXd13zA51IRyD5wb82_6e7Zx_4w5Cigy3GJ_Gzeo6fTG9L0fYtG1NPFhy
gkQNwi9m2

# LLMToolEmulator


Emulate tool execution using an LLM for testing purposes, replacing actual tool calls with AI-generated responses. LLM tool emulators are useful for the following:
 - Testing agent behavior without executing real tools.
 - Developing agents when external tools are unavailable or expensive.
 - Prototyping agent workflows before implementing actual tools.

In [51]:
from langchain.agents.middleware import LLMToolEmulator

emulated_agent = create_agent(
    model=model_or_paid_gpt56_luna_pro,
    tools=cinebot_tools,
    middleware=[LLMToolEmulator(tools=["book_seats", "cancel_booking"], model=model_or_paid_gpt56_luna_pro)],
    # ^ model= is passed EXPLICITLY here on purpose -- see the note below.
)

In [52]:
result = emulated_agent.invoke({"messages": [("user", "Book 2 seats for Interstellar")]})
print(result)

{
    'messages': [
        HumanMessage(
            content='Book 2 seats for Interstellar',
            additional_kwargs={},
            response_metadata={},
            id='fd548a4d-9b41-42c5-aebf-20ca5fcf0650'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'data': 
'gAAAAABqeDMeoAzsR53F47TTg6s57N_F6BZ5olvFXZUewmSIiqVp-VEIZq6yZKiQmm0kor1MYWLaarqMCDMm_Yb72uUvSLS6EssgTWhZbPPB6J7kSx
0WxQgvl1JrjL8sDyl9yz7XpB1TmS7BDfMKZu3maIdwSvZPRFZBvC2h1IsHPrrAXZGF_YSSGKZGhbOzNGn2_KjF-uAsri64p-89eXzH-Mzz-ybhnHXg3
cGvbNI8uDUjARhqCeRx_oWJ6tYu6FN6jZXjzup_qmN72BpRF3cZ0uC-Zm64EotIvtWfwa3EGPBEfZzUac7SqZrFw7IthkNOrO_ahgWnxBi-3I6rP_ad
Q-r8qJeEMXCxTW-lhwKapFTcCC0KM48x5tQKNWzD5nx2nKcMt4FJVOBKsWs6KPumT8YgFm_PM4fci_QvIqWz5zc1V0VCBQ0e5QJNYRcb-XTM3N-7Rx1
lh15uH-TePEsb6UxeSfpzVsuyVPQG9fDWhMqpRch25LsathadbbNpxEFpOHnk3y0FV53KyJdDfbVJTF6TRYbjTDtTmk2otMOhV4DasIn0HY3WHPkKyn
dLR8_A7G9vjJzJKcKyOZiIbnaRn63No1HA6H7GH7zDDpRQJiBYxA7C6ZTxs9j6XY7on68geLYbRA_sa9SpDjlOUdiyJ1M3iqJQ59mLJ9BorTRhPrnSP
jxrhAmbBefDFCpXWSuMSxivbzKuHBus1N3Xxx8Y62KZRMzK41M9pu9bZ_xHd5bxF03RNH9fyou7YoppcwVNL_bwZvj15VaLTppm_BxBFbDhLizM_gPm
1UWV2F-2nbuKOKXyBVvIYbW9aPQRWtwKjvmYe2BAY4kHW1TQNzsCNw1ucASyP98fGVH5An6BEouUcxB7UE9UIvqMu4Nt9e_yXod8AGnSfOpQx6KV4vk
uec4CtAQuK21s3ttapT-5LpeVr9mks4mtiv4D1bzqeXucmJunpwQrCLN5CGxSuycjndD85L-5q4J-fVxSysva-AyT7rW7QUf9MBrLaMx-svyWUBhldC
1IHgJzrRadt0xiXA8oEnMH9bmDLUmMobvcE6e-5byd2QBG66qS6HaQgBuFhyqbivNzjMcwOKj8WUtYbdqYT2yehkMAXaNgoGRcvS-2GSWwaNI4ROKdI
F08iSgx9T7KijI9b9sRibz4vftIJmGOmAaSnxT2zQcgsViiaHcUtxGUG3PHO6Fhb9c-LqbYgPzpTiuxTnemBNczIHPoEz5w0ruLT3w1yEaKzROMs1sB
ioKjfAYkmqfk93Lc3LyzVi3bh21yBKAz99FwqZOTshmySssA-vMFWB_4VjRL5NrkKRur1ljDiBZgQtFsvI2YMaYddTaWKBXieabMIBeYMDbquiUcFUl
Gxc54Fp-bNW0_NF-1F_bJ0brH_z_kKlWG2zSTY0xKGdlpdxTxc0HKS8ZBP3gaJRM2YUcSPRZSKyRn9_BNPiy7Tdp5AwaTf91woFsSTcbcixEWbEAEYs
JzDTGgGTEqXWN067-ZAMRubtggUKOzAoT046_gIfXDsPAR3rX4ghcChJxTAbTg4PpPaRo5tnxbF2ab6t-qQ0bO5vUdvglcIKTkl0WiFcI8ew77SDoZU
o7pYX0orj6o2UQAn3ygkKc5DWeuNOxQFZZbtY8dLk6JOEXv2y0kunDsUTSXPdGQjG__tOk8mHMca_0Qk7--MutvDA==.eyJlbmRwb2ludF9zbHVnIjo
ib3BlbmFpL2dwdC01LjYtbHVuYS1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_0e5e5aad64ce5dd7016a78331e5d8081939eae7a2d3c6c23b4',
                        'index': 0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786262298-bJsjdLw0TniCuCsgUe40',
                'created': 1786262298,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0004952,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0002616,
                    'upstream_inference_prompt_cost': 0.0002336,
                    'upstream_inference_cost': 0.0004952
                }
            },
            id='lc_run--019fe587-9a66-7a03-bc93-771ccdb2e144-0',
            tool_calls=[
                {
                    'name': 'book_seats',
                    'args': {'movie_title': 'Interstellar', 'seat_count': 2},
                    'id': 'call_U8IyKqGpyz7TWRBhXqh7hXAw',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 2336,
                'output_tokens': 436,
                'total_tokens': 2772,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 340}
            }
        ),
        ToolMessage(
            content='{"status":"confirmed","movie_title":"Interstellar","seat_count":2,"seats":["H12","H13"],"booki
ng_id":"BK-739184","

In [53]:
import base64

In [61]:
print(result)

{
    'messages': [
        HumanMessage(
            content='Book 2 seats for Interstellar',
            additional_kwargs={},
            response_metadata={},
            id='fd548a4d-9b41-42c5-aebf-20ca5fcf0650'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'data': 
'gAAAAABqeDMeoAzsR53F47TTg6s57N_F6BZ5olvFXZUewmSIiqVp-VEIZq6yZKiQmm0kor1MYWLaarqMCDMm_Yb72uUvSLS6EssgTWhZbPPB6J7kSx
0WxQgvl1JrjL8sDyl9yz7XpB1TmS7BDfMKZu3maIdwSvZPRFZBvC2h1IsHPrrAXZGF_YSSGKZGhbOzNGn2_KjF-uAsri64p-89eXzH-Mzz-ybhnHXg3
cGvbNI8uDUjARhqCeRx_oWJ6tYu6FN6jZXjzup_qmN72BpRF3cZ0uC-Zm64EotIvtWfwa3EGPBEfZzUac7SqZrFw7IthkNOrO_ahgWnxBi-3I6rP_ad
Q-r8qJeEMXCxTW-lhwKapFTcCC0KM48x5tQKNWzD5nx2nKcMt4FJVOBKsWs6KPumT8YgFm_PM4fci_QvIqWz5zc1V0VCBQ0e5QJNYRcb-XTM3N-7Rx1
lh15uH-TePEsb6UxeSfpzVsuyVPQG9fDWhMqpRch25LsathadbbNpxEFpOHnk3y0FV53KyJdDfbVJTF6TRYbjTDtTmk2otMOhV4DasIn0HY3WHPkKyn
dLR8_A7G9vjJzJKcKyOZiIbnaRn63No1HA6H7GH7zDDpRQJiBYxA7C6ZTxs9j6XY7on68geLYbRA_sa9SpDjlOUdiyJ1M3iqJQ59mLJ9BorTRhPrnSP
jxrhAmbBefDFCpXWSuMSxivbzKuHBus1N3Xxx8Y62KZRMzK41M9pu9bZ_xHd5bxF03RNH9fyou7YoppcwVNL_bwZvj15VaLTppm_BxBFbDhLizM_gPm
1UWV2F-2nbuKOKXyBVvIYbW9aPQRWtwKjvmYe2BAY4kHW1TQNzsCNw1ucASyP98fGVH5An6BEouUcxB7UE9UIvqMu4Nt9e_yXod8AGnSfOpQx6KV4vk
uec4CtAQuK21s3ttapT-5LpeVr9mks4mtiv4D1bzqeXucmJunpwQrCLN5CGxSuycjndD85L-5q4J-fVxSysva-AyT7rW7QUf9MBrLaMx-svyWUBhldC
1IHgJzrRadt0xiXA8oEnMH9bmDLUmMobvcE6e-5byd2QBG66qS6HaQgBuFhyqbivNzjMcwOKj8WUtYbdqYT2yehkMAXaNgoGRcvS-2GSWwaNI4ROKdI
F08iSgx9T7KijI9b9sRibz4vftIJmGOmAaSnxT2zQcgsViiaHcUtxGUG3PHO6Fhb9c-LqbYgPzpTiuxTnemBNczIHPoEz5w0ruLT3w1yEaKzROMs1sB
ioKjfAYkmqfk93Lc3LyzVi3bh21yBKAz99FwqZOTshmySssA-vMFWB_4VjRL5NrkKRur1ljDiBZgQtFsvI2YMaYddTaWKBXieabMIBeYMDbquiUcFUl
Gxc54Fp-bNW0_NF-1F_bJ0brH_z_kKlWG2zSTY0xKGdlpdxTxc0HKS8ZBP3gaJRM2YUcSPRZSKyRn9_BNPiy7Tdp5AwaTf91woFsSTcbcixEWbEAEYs
JzDTGgGTEqXWN067-ZAMRubtggUKOzAoT046_gIfXDsPAR3rX4ghcChJxTAbTg4PpPaRo5tnxbF2ab6t-qQ0bO5vUdvglcIKTkl0WiFcI8ew77SDoZU
o7pYX0orj6o2UQAn3ygkKc5DWeuNOxQFZZbtY8dLk6JOEXv2y0kunDsUTSXPdGQjG__tOk8mHMca_0Qk7--MutvDA==.eyJlbmRwb2ludF9zbHVnIjo
ib3BlbmFpL2dwdC01LjYtbHVuYS1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_0e5e5aad64ce5dd7016a78331e5d8081939eae7a2d3c6c23b4',
                        'index': 0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786262298-bJsjdLw0TniCuCsgUe40',
                'created': 1786262298,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0004952,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0002616,
                    'upstream_inference_prompt_cost': 0.0002336,
                    'upstream_inference_cost': 0.0004952
                }
            },
            id='lc_run--019fe587-9a66-7a03-bc93-771ccdb2e144-0',
            tool_calls=[
                {
                    'name': 'book_seats',
                    'args': {'movie_title': 'Interstellar', 'seat_count': 2},
                    'id': 'call_U8IyKqGpyz7TWRBhXqh7hXAw',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 2336,
                'output_tokens': 436,
                'total_tokens': 2772,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 340}
            }
        ),
        ToolMessage(
            content='{"status":"confirmed","movie_title":"Interstellar","seat_count":2,"seats":["H12","H13"],"booki
ng_id":"BK-739184","

In [57]:
messages = result["messages"]

In [70]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator
from langchain.tools import tool


@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location."""
    return f"Weather in {location}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email."""
    return "Email sent"


# Emulate all tools (default behavior)
agent = create_agent(
    model=model_or_paid_gpt56_luna_pro,
    tools=[get_weather, send_email],
    middleware=[LLMToolEmulator(model=model_or_paid_gpt56_luna_pro,)],
)


In [71]:
result = agent.invoke({"messages": [("user", "Please send a email to my manager for leave tomorrow by mentioning the bad weather in Gurgaon")]})

print(result)

{
    'messages': [
        HumanMessage(
            content='Please send a email to my manager for leave tomorrow by mentioning the bad weather in 
Gurgaon',
            additional_kwargs={},
            response_metadata={},
            id='a75d6b05-0037-41d8-a56d-6975458f6dca'
        ),
        AIMessage(
            content='Please provide your manager’s email address and confirm whether you’re requesting a full-day 
or half-day leave. I’ll then send the email mentioning the bad weather in Gurgaon.',
            additional_kwargs={
                'reasoning_content': "**Clarifying details for email**\n\nI need to figure out the manager's email,
and I might need to clarify the date, which is currently unknown. I should ask for the email and maybe draft it as 
well since the user wants it sent. I need the recipient's address and possibly their name, and I should ask 
concisely. The user mentioned bad weather, so I can include that generally, but I won't use the weather tool unless
necessary. I’ll also confirm the exact leave date.",
                'reasoning_details': [
                    {
                        'summary': "**Clarifying details for email**\n\nI need to figure out the manager's email, 
and I might need to clarify the date, which is currently unknown. I should ask for the email and maybe draft it as 
well since the user wants it sent. I need the recipient's address and possibly their name, and I should ask 
concisely. The user mentioned bad weather, so I can include that generally, but I won't use the weather tool unless
necessary. I’ll also confirm the exact leave date.",
                        'type': 'reasoning.summary',
                        'format': 'openai-responses-v1',
                        'index': 0
                    },
                    {
                        'data': 
'gAAAAABqeEMXiP-8Lg4T9tuXkxf_0hNTuj0HdLTutl_I_wNvgbQJ3LFT3zf3T_B50ZO3-bEHG1vCDZotioMldz4HKNodpkcdoDKWgwD1XjgdfdgiPs
XQbC9kxwTLAa8LkT8ZfYz0fvGFbCjHi7Diqd--irhV8rFa5fqva9L_4lFWw26Ezuc8PCwdAI2GmhlykJIsHLWP15q3-PA-6K_P-FA3JvwaG8kKrMHl7
MHYHHZbZVYBa1xCX_lQ3xbycW7WF-G0KYsIFR4qyTZ8sBC-YXggjrCTXGrxED4pv8jyVY0hwQAQNxG4kr5lk6ZPCMlPXX8UcZwsI29KyI-P7hB2mNRc
9C7zb7gXz3dVwRa0M5QfBZ5mmizt8Kq0i2fANZK0EUmwLKY_Bbv0eM7rwc4iKWqFAbCJO_v0IIMjKtJPnm7cHgRCBp-hnaSPIVJjc1gtv4AnkSedhCL
ySM3MgsTrrAWGKW7-261u5BPiyKhoupsG0W7x3IJLtsTh5qep3QpDplnqcWVsoJehPWONvsn14AgbR17uMpXTisKOdxi8l40D2pdZsVtoC_j70r1fDL
VKcoiwCumuFJkSDU72DEOWkTRCEP1fGh_Ij_6eD31n7HlJPOcKAvRUN9PWPNJeixzr112KOgmo4GyFyareTk6Ew1BHd89tpzHhKaTUI0-x9YFtgtX85
A60NHFEGyu6RrkN0iSWxREYjz4km7v4jo8XmLmBW_Gd4cDLp5O_IUolqvgZwIT7suTlyLTQ9jcaAs3oPmU6drJHfjqG4te_hPi0OY0lcFlBv7oh-aJY
0vs1Md2UUKLT4mGmEPgoAI1pzsdfbf9mJbgTynKKehAUEiO53OGc-Tc0_Plaj08FyWU_b_qR0Hh-ej3jHcaQ1eNQHBJFvxvO0YCYidufwWivrd7WmDF
RqVU2osCt0Wkv6zeNxQvY9e64KXYFslDUR609yWbijBfVQYQyKPOpds4z9AVCZlf8DJUWJ4wsys7hKkTIHj78EDydwi0nGMR8csOzuVqjGtee1YL5uU
kFDywtWCm3-BZnSMXWGAaSG7kiKceqjDnNIotSl4qm1p44XluSQRZVMocs1lVIrE0eyXAj5nbhUf6IYFUzYc_Oa8wNzEleiDfr_qlHsg7XUPVHntqqK
zr6GJyo-yM8ZT_1BaHSgF7AwblRiz-gD_t2aU3j2jsBE1lat_Uh3apLKlaH_X35_z4C3x3NnNWmbIDDSqkzT8QX0zyazf5GDjn8RHRMM15o17UbfkXq
dfm7FuRBixMkYiBYchOwN6pP79-iEeIAfTD1KpdADKTgMasDVKeNJyoBBZ0CUVOfA3-k7dCebwExLgEgcUVOUdnBZKjsY8hk_NeIm2EA6dZtPtn8bxH
LUa987ocWLHEUjRPeioY1_rnFtENOFsbzeaJhy6SE_xFI8QWPOi08rkmaYKg260ea5mtE_JrYe9D5pRmZ5FiYNtC4KEx9XF-E8MYlrD22D_7hQU_F89
dOh6Eg-MZqGDd88mYh8qe_dt6K0pZA5ZtjcYBdGRuRc62kiIBMMtTBNDnKXjC_vFMdN_AsYAjaNg==.eyJlbmRwb2ludF9zbHVnIjoib3BlbmFpL2dw
dC01LjYtbHVuYS1wcm8tMjAyNjA3MDl8b3BlbmFpIn0',
                        'type': 'reasoning.encrypted',
                        'format': 'openai-responses-v1',
                        'id': 'rs_0c3e07d68bc31fa4016a78431712b48195a0db8c7c59174cb3',
                        'index': 1
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1786266384-uhaPJMIkEsn0DlJq5oG6',
                'created': 1786266384,
     